In [ ]:
import re
import numpy as np
import pandas as pd
import gensim.corpora as corpora
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split, GridSearchCV
import matplotlib.pyplot as plt
import seaborn as sns

# Bayesian approach 

In this section we implement the Topic Model using Latent Dirichlet Allocation

In [ ]:
def load_pottery_data(filename, culture_name):
    documents = []
    with open(filename, 'r') as f:
        for line_num, line in enumerate(f, 1):
            triplets = re.findall(r"\('([^']+)', '([^']+)', '([^']+)'\)", line.strip())
            if triplets:
                doc_triplets = [f"({t[0]}, {t[1]}, {t[2]})" for t in triplets]
                documents.append({
                    'vessel_id': f"{culture_name}_{line_num}",
                    'culture': culture_name,
                    'triplets': doc_triplets,
                    'triplets_text': ' '.join(doc_triplets),
                    'num_triplets': len(doc_triplets)
                })
    return documents

# Loading all cultural datasets
print("Loading archaeological datasets...")
cw_docs = load_pottery_data('./data/clustered_paths_CW.csv', 'CW')
mi_docs = load_pottery_data('./data/clustered_paths_MI.csv', 'MI')
efg_docs = load_pottery_data('./data/clustered_paths_EFG.csv', 'EFG')
lfg_docs = load_pottery_data('./data/clustered_paths_LFG.csv', 'LFG')
fbw_docs = load_pottery_data('./data/clustered_paths_FBW.csv', 'FBW')
print("Datasets loaded.")

print(f"Dataset summary:")
print(f"  CW (Corded Ware): {len(cw_docs)} vessels")
print(f"  MI (Middle phase): {len(mi_docs)} vessels")
print(f"  EFG (Early/Final phase): {len(efg_docs)} vessels")
print(f"  LFG (Late phase): {len(lfg_docs)} vessels")

# All docs are combined to create the corpus, from the overall corpus we extract the clusters and topics later and compare at the end the distribution of words in each culture 
# extracting the empirical frequencies from the data from each separate culture and the one obtained from the topic modeling from the whole pool

NUM_CULTURES = 4

if NUM_CULTURES == 4:
    all_docs = cw_docs[:-2] + mi_docs[:-2] + efg_docs[:-2] + lfg_docs[:-2]
elif NUM_CULTURES == 2:
    all_docs = cw_docs + fbw_docs
cultures = [doc['culture'] for doc in all_docs]
culture_counts = Counter(cultures)

print(f"\nTotal vessels: {len(all_docs)}")
print(f"Culture distribution: {dict(culture_counts)}")

print("\nSample documents:")
for doc in all_docs[:10]:
    print(f"Vessel ID: {doc['vessel_id']}, Culture: {doc['culture']}, Num Triplets: {doc['num_triplets']}")
    print(f"Triplets: {doc['triplets_text']}\n")
from collections import Counter

# vocabulary extraction
def extract_vocabulary(docs):
    """Extract vocabulary and term frequencies from documents"""
    vocab_counter = Counter()
    for doc in docs:
        vocab_counter.update(doc['triplets'])
    vocabulary = list(vocab_counter.keys())
    term_frequencies = dict(vocab_counter)
    return vocabulary, term_frequencies

vocabulary, term_frequencies = extract_vocabulary(all_docs)
print(f"Extracted vocabulary of size {len(vocabulary)}")
print("Sample vocabulary terms and their frequencies:")
for term, freq in list(term_frequencies.items())[:10]:
    print(f"  {term}: {freq}")

In [ ]:
# preprocessing data - removing the most common triplets that do not add information to the model
triplet_counter = Counter()
for doc in all_docs:
    triplet_counter.update(doc['triplets'])

most_common_triplets = triplet_counter.most_common(5)
print(f"Most common triplets to remove: {most_common_triplets}")
for doc in all_docs:
    doc['triplets'] = [t for t in doc['triplets'] if t not in dict(most_common_triplets)]

# print triplet counts after removal
triplet_counter_after = Counter()
for doc in all_docs:
    triplet_counter_after.update(doc['triplets'])
print(f"Triplet counts after removal: {triplet_counter_after.most_common(10)}")

# remove them from the vocabulary as well
vocabulary = [t for t in vocabulary if t not in dict(most_common_triplets)]
print(f"Vocabulary size after removal: {vocabulary}")  


In [ ]:
print(f"Vocabulary size after removal: {len(vocabulary)}")

In [ ]:
import numpy as np
# by bootstrapping the documents we can create a larger dataset, and avoid the problem of words that appears only ones in the whole corpus
# Sampling with replacement groups of vessels from the same culture and combining their triplets, so for the LLN the distribution of words should converge to the "real" one
def create_resampled_documents(docs, group_size=3, num_new_docs=100):
    resampled_docs = []
    culture_groups = defaultdict(list)
    for doc in docs:
        culture_groups[doc['culture']].append(doc)
    
    for culture, culture_docs in culture_groups.items():
        for _ in range(num_new_docs):
            group = np.random.choice(culture_docs, size=group_size, replace=True)
            combined_triplets = []
            vessel_ids = []
            for vessel in group:
                combined_triplets.extend(vessel['triplets'])
                vessel_ids.append(vessel['vessel_id'])
            
            if combined_triplets:
                resampled_docs.append({
                    'vessel_id': f"RESAMPLED_{culture}_{len(resampled_docs)+1}",
                    'original_vessels': vessel_ids,
                    'culture': culture,
                    'triplets': combined_triplets,
                    'triplets_text': ' '.join(combined_triplets),
                    'num_triplets': len(combined_triplets),
                    'group_size': len(group)
                })
    
    return resampled_docs

print("=== BOOTSTRAP ===")
resampled_3 = create_resampled_documents(all_docs, group_size=3)
# Resample_3 is a dataset that has the following fields: Vessel_id, original_vessels, culture, triplets, num_triplets, group_size , 'triplets_text'
print("Vessel ID, Original Vessels, Culture, Num Triplets, Group Size")
for doc in resampled_3:
    print(f"{doc['vessel_id']}, {doc['original_vessels']}, {doc['culture']}, {doc['num_triplets']}, {doc['group_size']}")
    print(f"  Triplets: {doc['triplets_text']}")
resampled_5 = create_resampled_documents(all_docs, group_size=7)
resampled_15 = create_resampled_documents(all_docs, group_size=15)

print(f"Original dataset: {len(all_docs)} vessels")
print(f"Resampled (group_size=3): {len(resampled_3)} documents")
print(f"Resampled (group_size=5): {len(resampled_5)} documents")

print(f"Resampled (group_size=5): {len(resampled_15)} documents")
# Compare average document lengths
datasets_comparison = [
    ("Original", all_docs),
    ("Resampled-3", resampled_3),
    ("Resampled-5", resampled_5),
    ("Resampled-15", resampled_15)
]

print(f"\nEffect of resampling on document characteristics:")
for name, docs in datasets_comparison:
    avg_length = np.mean([doc['num_triplets'] for doc in docs])
    cultures_in_dataset = [doc['culture'] for doc in docs]
    culture_balance = dict(Counter(cultures_in_dataset))
    print(f"  {name}:")
    print(f"    Average triplets per document: {avg_length:.2f}")
    print(f"    Culture distribution: {culture_balance}")

In [ ]:
import re
from collections import Counter
from scipy.sparse import csr_matrix
import numpy as np

def normalize_triplet(t: str) -> str:
    """
    Canonical form: (A, B, C) with single spaces after commas.
    Accepts variants with quotes.
    """
    t = t.strip().strip('"').strip("'")
    # remove internal single quotes
    t = t.replace("',", ",").replace("'", "")
    # collapse whitespace
    t = re.sub(r"\s+", " ", t)
    # optional: enforce space after commas
    parts = [p.strip() for p in t.strip("()").split(",")]
    if len(parts) == 3:
        return f"({parts[0]}, {parts[1]}, {parts[2]})"
    return t


external_vocab_raw = vocabulary  # or use another list if you prefer
external_vocab_norm = [normalize_triplet(v) for v in external_vocab_raw]
# Deduplicate while preserving order
seen = set()
external_vocab = [v for v in external_vocab_norm if not (v in seen or seen.add(v))]
vocab_index = {tok: i for i, tok in enumerate(external_vocab)}
print("[info] Built external vocabulary index.")
print(vocab_index)

print(f"[info] External vocabulary size: {len(external_vocab)}")

def build_dtm_with_external_vocab(docs, vocab_index):
    """
    Build a CSR document-term matrix using only tokens in vocab_index.
    Counts = frequency of triplet in document.
    """
    rows, cols, data = [], [], []
    doc_lengths = []
    for d, doc in enumerate(docs):
        counts = Counter(normalize_triplet(t) for t in doc['triplets'])
        doc_lengths.append(sum(counts.values()))
        for tok, c in counts.items():
            idx = vocab_index.get(tok)
            if idx is not None:
                rows.append(d)
                cols.append(idx)
                data.append(c)
    X = csr_matrix((data, (rows, cols)), shape=(len(docs), len(vocab_index)), dtype=np.int32)
    plt.figure(figsize=(8, 6))
    plt.spy(X, markersize=1)
    plt.show()
    return X, doc_lengths

datasets = {
    'original': all_docs,
    'resampled_3': resampled_3,
    'resampled_5': resampled_5, 
    'resampled_15': resampled_15
}

matrices = {}
for name, docs_set in datasets.items():
    X, lens = build_dtm_with_external_vocab(docs_set, vocab_index)
    matrices[name] = {
        'X': X,
        'features': np.array(external_vocab),
        'docs': docs_set,
        'cultures': [d['culture'] for d in docs_set]
    }
    nnz = X.nnz
    density = nnz / (X.shape[0] * X.shape[1]) if X.shape[0] and X.shape[1] else 0
    print(f"\n{name.upper()} dataset:")
    print(f"  Documents: {X.shape[0]}")
    print(f"  Vocabulary (external): {X.shape[1]}")
    print(f"  Non-zero entries: {nnz}")
    print(f"  Density: {density:.6f}")

print("\nVocabulary example (first 10):")
for i, feat in enumerate(external_vocab[:10]):
    print(f"  {i+1:2d}. {feat}")

def coverage_report(doc_triplets, vocab_index):
    norm = [normalize_triplet(t) for t in doc_triplets]
    in_vocab = sum(1 for t in norm if t in vocab_index)
    return in_vocab, len(norm), in_vocab / len(norm) if norm else 0.0

if 'new_sample_doc' in globals():
    print("\n[coverage] New sample docs:")
    for i, dlist in enumerate(new_sample_doc, 1):
        inv, tot, cov = coverage_report(dlist, vocab_index)
        print(f"  Doc {i}: {inv}/{tot} in vocab ({cov:.1%})")

print(vocabulary)

# Gensim parenthesis

In [ ]:

# create also the vocabulary gensim dictionary and corpus for LDA
def create_gensim_corpus(docs):
    """Create gensim dictionary and corpus from documents"""
    texts = [doc['triplets'] for doc in docs]
    dictionary = corpora.Dictionary(texts)
    corpus = [dictionary.doc2bow(text) for text in texts]
    return dictionary, corpus

# print the gensim dictionary and corpus size for the original dataset
dictionary_gensim, corpus_gensim = create_gensim_corpus(resampled_3)
print(f"\nGensim dictionary size (original dataset): {len(dictionary_gensim)}")
print(f"Gensim corpus size (original dataset): {len(corpus_gensim)}")

# print some samples in the corpus to see how it looks like
print("\nSample corpus entries (first 5 documents):")
for i, doc_bow in enumerate(corpus_gensim[:5]):
    print(f"Document {i+1}: {doc_bow}")
    # convert back to triplets for clarity
    triplets = [dictionary_gensim[id] for id, freq in doc_bow]
    print(f"  Triplets: {triplets}")


In [ ]:
# LDA Gensim external lybrary 
from gensim.models import LdaModel
lda_model_gensim = LdaModel(
    corpus=corpus_gensim,
    id2word=dictionary_gensim,
    num_topics=NUM_CULTURES,
    random_state=42, 
    alpha=0.1,   
    eta=1,     
    iterations=1000  
)

In [ ]:
print("\nTop words for each topic:")
topics_gensim = lda_model_gensim.print_topics(num_words=5) # Get top 5 words per topic
for topic in topics_gensim:
    print(topic)


In [ ]:
#empirical distribution of words in each culture
def get_empirical_word_distribution(docs, dictionary):
    culture_word_counts = defaultdict(Counter)
    for doc in docs:
        culture = doc['culture']
        for triplet in doc['triplets']:
            if triplet in dictionary.token2id:
                culture_word_counts[culture][triplet] += 1
    
    # Convert counts to probabilities
    culture_word_distributions = {}
    for culture, counter in culture_word_counts.items():
        total_count = sum(counter.values())
        culture_word_distributions[culture] = {word: count / total_count for word, count in counter.items()}
    
    return culture_word_distributions

empirical_distributions = get_empirical_word_distribution(all_docs, dictionary_gensim)
empirical_CW = empirical_distributions.get('CW', {})
print(len(empirical_CW.values()))
# count how many vocabulary there are in cw_docs
dictionary_CW = set()
for doc in cw_docs:
    dictionary_CW.update(doc['triplets'])
print(len(dictionary_CW))

dictionary_FBW = set()
for doc in fbw_docs:
    dictionary_FBW.update(doc['triplets'])
print(len(dictionary_FBW))

print(len(set(dictionary_CW).intersection(set(dictionary_FBW))))
empirical_MI = empirical_distributions.get('MI', {})
empirical_EFG = empirical_distributions.get('EFG', {})
empirical_LFG = empirical_distributions.get('LFG', {})

print("\nEmpirical word distribution samples:")
print("CW:", dict(list(empirical_CW.items())[:5]))
# plot the word-topic distribution for each document 
topic_word_distributions = lda_model_gensim.get_topics()
print(topic_word_distributions.shape)

for _ in range(NUM_CULTURES):
    plt.figure(figsize=(10, 20))
    plt.barh(range(len(topic_word_distributions[_])), topic_word_distributions[_])
    plt.barh(range(len(empirical_CW.values())), list(empirical_CW.values()), alpha=0.5, color='orange')
    plt.title(f"Topic {_} Word Distribution")
    plt.xlabel("Probability")
    plt.ylabel("Word Index")
    plt.yticks(range(len(topic_word_distributions[_])), [dictionary_gensim[i] for i in range(len(topic_word_distributions[_]))])
    plt.show()


In [ ]:
from gensim.models import CoherenceModel

# Compute Coherence Score (c_v)
preprocessed_docs = [doc['triplets'] for doc in all_docs]
coherence_model_lda = CoherenceModel(model=lda_model_gensim, texts=preprocessed_docs, dictionary=dictionary_gensim, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print(f'\nCoherence Score (c_v): {coherence_lda:.4f}')

# LDA Implementation scratch

In [ ]:
class LDA_manual:
    
    def __init__(self, n_topics=4, alpha=1, beta=1, n_iter=100, random_state=42):
        self.n_topics = n_topics
        self.alpha = alpha  # Document-topic prior
        self.beta = beta    # Topic-word prior  
        self.n_iter = n_iter
        self.random_state = random_state
        np.random.seed(random_state)
        
    def fit(self, X, feature_names=None): # Collapsed Gibbs Sampling feature_names is the vocabulary
        """
        Run the collapsed gibbs sampling on a document-word matrix X 
        """
        self.n_docs, self.n_words = X.shape
        
        if hasattr(X, 'toarray'):
            X = X.toarray()
        # persist vocabulary order
        if feature_names is not None:
            if len(feature_names) != self.n_words:
                raise ValueError("feature_names length mismatch matrix width")
            self.feature_names_ = np.array(feature_names, dtype=object)
        else:
            self.feature_names_ = np.arange(self.n_words, dtype=object)


        self.doc_topic_counts = np.zeros((self.n_docs, self.n_topics))
        self.topic_word_counts = np.zeros((self.n_topics, self.n_words))
        self.topic_counts = np.zeros(self.n_topics)
        
        # For each document and word, assign random topic (it will be adjusted using gibbs sampling)
        self.topic_assignments = []
        for d in range(self.n_docs):
            doc_assignments = []
            for w in range(self.n_words):
                word_count = int(X[d, w]) 
                for _ in range(word_count):
                    topic = np.random.randint(0, self.n_topics)
                    doc_assignments.append(topic)
                    self.doc_topic_counts[d, topic] += 1
                    self.topic_word_counts[topic, w] += 1
                    self.topic_counts[topic] += 1
            self.topic_assignments.append(doc_assignments)
        
        print(f"Running {self.n_iter} Gibbs sampling iterations...")
        for iteration in range(self.n_iter):
            if iteration % 20 == 0:
                print(f"  Iteration {iteration}")
            
            # Resample topic assignments with new probability P(z_w|z_-w, D)
            # assign to each word a topic (first randomly, then following p, that takes into account the times that a word is associated to a certain topic )
            for d in range(self.n_docs):
                word_idx = 0
                for w in range(self.n_words):
                    word_count = int(X[d, w])
                    for _ in range(word_count):
                        if word_idx < len(self.topic_assignments[d]):
                            # Remove current assignment
                            old_topic = self.topic_assignments[d][word_idx]
                            self.doc_topic_counts[d, old_topic] -= 1
                            self.topic_word_counts[old_topic, w] -= 1
                            self.topic_counts[old_topic] -= 1
                            
                            p = np.zeros(self.n_topics)
                            for k in range(self.n_topics):
                                p[k] = ((self.doc_topic_counts[d, k] + self.alpha)) * ((self.topic_word_counts[k, w] + self.beta) / (self.topic_counts[k] + self.n_words * self.beta)) 
                                # /(np.sum(self.doc_topic_counts[d]+len(self.feature_names_)*self.alpha)) does not need because is constant and so it can be putted in the normalisation 
                            
                            p = p / np.sum(p)
                            new_topic = np.random.choice(self.n_topics, p=p) # Sample new topic for that word
            
                            self.topic_assignments[d][word_idx] = new_topic
                            self.doc_topic_counts[d, new_topic] += 1
                            self.topic_word_counts[new_topic, w] += 1
                            self.topic_counts[new_topic] += 1
                            
                        word_idx += 1
        
        # Final expected values for topic distributions
        self.doc_topic_dist = (self.doc_topic_counts + self.alpha) / \
                             (self.doc_topic_counts.sum(axis=1, keepdims=True) + self.n_topics * self.alpha)
        
        self.topic_word_dist = (self.topic_word_counts + self.beta) / \
                              (self.topic_word_counts.sum(axis=1, keepdims=True) + self.n_words * self.beta)
        
        return self
    
    def get_top_words(self, feature_names, n_words=10):
        """Get top words for each topic"""
        top_words = []
        for topic_idx in range(self.n_topics):
            top_indices = self.topic_word_dist[topic_idx].argsort()[-n_words:][::-1]
            topic_words = [(feature_names[i], self.topic_word_dist[topic_idx, i]) 
                          for i in top_indices]
            top_words.append(topic_words)
        return top_words

    def transform(self, X, n_iter=50, feature_names=None):
        """
        Infer per-document topic distributions for X (fold-in) using fixed phi. (so given a trained model we can infer the topic distribution for new documents)

        """
        if not hasattr(self, "topic_word_dist"):
            raise AttributeError("Model not fitted. Call fit first.")
        V_model = len(self.feature_names_)
        if feature_names is not None:
            feature_names = np.array(feature_names, dtype=object)
            if X.shape[1] != len(feature_names):
                raise ValueError("X width must match provided feature_names length")
            # Build aligned matrix with model's vocab order
            col_pos = {f: i for i, f in enumerate(feature_names)}
            cols = [col_pos.get(f, None) for f in self.feature_names_]
            # Build new matrix (CSR safe gather)
            from scipy.sparse import csr_matrix, issparse
            if any(c is None for c in cols):
                # columns missing in X → treat as zeros
                take = np.array([c for c in cols if c is not None])
                X_aligned = X[:, take]
                # pad missing columns with zeros at the right places
                if issparse(X):
                    X_full = csr_matrix((X.shape[0], V_model), dtype=X.dtype)
                else:
                    X_full = np.zeros((X.shape[0], V_model), dtype=X.dtype)
                # place available columns
                dest = np.array([i for i, c in enumerate(cols) if c is not None])
                X_full[:, dest] = X_aligned
                X = X_full
            else:
                X = X[:, np.array(cols)]
        else:
            # Require same width if no names given
            if X.shape[1] != V_model:
                raise ValueError("X has different number of columns than model vocabulary. Provide feature_names to align.")

        # Convert to dense for simple inference
        if hasattr(X, "toarray"):
            X = X.toarray()

        K = self.n_topics
        phi = self.topic_word_dist  # (K, V)
        D = X.shape[0]
        # Initialize theta uniformly or use the dirichelet prior
        uniform_init = False
        if uniform_init:
            theta = np.full((D, K), 1.0 / K)
        else:
            theta = np.random.dirichlet(1.0 /  np.ones(K), size=D)

        # fixed-point iterations: p(k|w) ∝ phi[w|k] * theta[d,k] mantaining phi fixed from the training
        for _ in range(n_iter):
            for d in range(D):
                x = X[d]  # (V,)
                if x.sum() == 0:
                    theta[d] = 1.0 / K
                    continue
                r = phi * theta[d][:, None]
                r /= (r.sum(axis=0, keepdims=True) + 1e-12)
                # expected topic counts in doc d
                n_k = (r * x[None, :]).sum(axis=1)  # (K,)
                # add Dirichlet prior and normalize
                theta[d] = (n_k + self.alpha)
                theta[d] /= theta[d].sum()
        return theta

    def predict_proba(self, X, **kwargs):
        return self.transform(X, **kwargs)

    def predict(self, X, **kwargs):
        return self.transform(X, **kwargs).argmax(axis=1)

In [ ]:
# Test LDA from scratch
print("LDA FROM SCRATCH ")
X_original = matrices['original']['X']
features_original = matrices['original']['features']

X_resampled_3 = matrices['resampled_3']['X']
features_resampled_3 = matrices['resampled_3']['features']  

X_resampled_5 = matrices['resampled_5']['X']
features_resampled_5 = matrices['resampled_5']['features']

X_resampled_15 = matrices['resampled_15']['X']
features_resampled_15 = matrices['resampled_15']['features']


In [ ]:
# explore the X matrix access to the documents 
print(X_resampled_3[:,90])

In [ ]:
print("Fitting LDA with 4 topics from scratch...")
lda_scratch = LDA_manual(n_topics=NUM_CULTURES, alpha = 1, beta = 1, n_iter=200, random_state=42)
lda_scratch.fit(X_resampled_5, feature_names=features_resampled_5)
print("\nTop techniques per topic (LDA from scratch):")
top_words = lda_scratch.get_top_words(features_resampled_5, n_words=10)
for i, topic_words in enumerate(top_words):
    print(f"\nTopic {i}:")
    for word, weight in topic_words:
        print(f"  {word}: {weight:.4f}")

In [ ]:
# plot the topic word distributions for each culture on the same plot with different colours
import matplotlib.pyplot as plt

phi = lda_scratch.topic_word_dist

# empirical distributions for each culture
MI_triplets = []
triplets_per_vessel = []
for doc in mi_docs:
    MI_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
MI_counts = Counter(MI_triplets)
MI_total = sum(MI_counts.values())
MI_dist = np.array([MI_counts.get(triplet, 0) / MI_total for triplet in features_original])

EFG_triplets = []
triplets_per_vessel = []
for doc in efg_docs:
    EFG_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
EFG_counts = Counter(EFG_triplets)
EFG_total = sum(EFG_counts.values())
EFG_dist = np.array([EFG_counts.get(triplet, 0) / EFG_total for triplet in features_original])

LFG_triplets = []
triplets_per_vessel = []
for doc in lfg_docs:
    LFG_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
LFG_counts = Counter(LFG_triplets)
LFG_total = sum(LFG_counts.values())
print("LFG total:", LFG_total)
print("LFG counts for original features:", [LFG_counts.get(triplet, 0) for triplet in features_original])
LFG_dist = np.array([LFG_counts.get(triplet, 0) / LFG_total for triplet in features_original])  

CW_triplets = []
triplets_per_vessel = []
for doc in cw_docs:
    CW_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
CW_counts = Counter(CW_triplets)
CW_total = sum(CW_counts.values())
CW_dist = np.array([CW_counts.get(triplet, 0) / CW_total for triplet in features_original])


# calculate using the JS divergence the similarity between the empirical distribution and the topic distribution for each culture
# which is the best matching between empirical and topic distribution
from scipy.spatial.distance import jensenshannon    
def js_divergence(p, q):
    return jensenshannon(p, q)**2  # square to get the divergence   
js_MI = [js_divergence(MI_dist, phi[k]) for k in range(NUM_CULTURES)]
js_CW = [js_divergence(CW_dist, phi[k]) for k in range(NUM_CULTURES)]
js_EFG = [js_divergence(EFG_dist, phi[k]) for k in range(NUM_CULTURES)]
js_LFG = [js_divergence(LFG_dist, phi[k]) for k in range(NUM_CULTURES)]

print("\nJensen-Shannon Divergence between empirical and topic distributions:")
print(f"MI: {js_MI}")
print(f"CW: {js_CW}")
print(f"EFG: {js_EFG}")
print(f"LFG: {js_LFG}") 

MI_index = np.argmin(js_MI)
CW_index = np.argmin(js_CW) 
EFG_index = np.argmin(js_EFG)
LFG_index = np.argmin(js_LFG)

culture_mapping = {
    'MI': MI_index,
    'CW': CW_index,
    'EFG': EFG_index,
    'LFG': LFG_index
}

plt.figure(figsize=(12, 15))
plt.barh(range(len(features_original)), MI_dist, alpha=0.5, label='Empirical MI Distribution')
plt.barh(range(len(features_original)), phi[MI_index], alpha=0.5, label=f'Topic {MI_index}')
plt.xlabel('Probability')
plt.yticks(range(len(features_original)), features_original)
plt.title(f'Empirical Triplet Distribution for MI Culture alpha = {lda_scratch.alpha} beta = {lda_scratch.beta}')
plt.legend()
plt.tight_layout()
plt.savefig(f'./Prior_exploration/MI_topic_word_distr_alpha{1}_beta{1}.png')
plt.show()

plt.figure(figsize=(12, 15))
plt.barh(range(len(features_original)), CW_dist, alpha=0.5, label='Empirical CW Distribution')
plt.barh(range(len(features_original)), phi[CW_index], alpha=0.5, label=f'Topic {CW_index}')
plt.xlabel('Probability')
plt.yticks(range(len(features_original)), features_original)
plt.title(f'Empirical Triplet Distribution for CW Culture alpha = {lda_scratch.alpha} beta = {lda_scratch.beta}')
plt.legend()
plt.tight_layout()
plt.savefig(f'./Prior_exploration/CW_topic_word_distr_alpha{1}_beta{1}.png')
plt.show()

plt.figure(figsize=(12, 15))
plt.barh(range(len(features_original)), EFG_dist, alpha=0.5, label='Empirical EFG Distribution')
plt.barh(range(len(features_original)), phi[EFG_index], alpha=0.5, label=f'Topic {EFG_index}')
plt.xlabel('Probability')
plt.yticks(range(len(features_original)), features_original)
plt.title(f'Empirical Triplet Distribution for EFG Culture alpha = {lda_scratch.alpha} beta = {lda_scratch.beta}')
plt.legend()
plt.tight_layout()
plt.savefig(f'./Prior_exploration/EFG_topic_word_distr_alpha{1}_beta{1}.png')
plt.show()  

plt.figure(figsize=(12, 15))
plt.barh(range(len(features_original)), LFG_dist, alpha=0.5, label='Empirical LFG Distribution')
plt.barh(range(len(features_original)), phi[LFG_index], alpha=0.5, label=f'Topic {LFG_index}')
plt.xlabel('Probability')
plt.yticks(range(len(features_original)), features_original)
plt.title(f'Empirical Triplet Distribution for LFG Culture alpha = {lda_scratch.alpha} beta = {lda_scratch.beta}')
plt.legend()
plt.tight_layout()
plt.savefig(f'./Prior_exploration/LFG_topic_word_distr_alpha{1}_beta{1}.png')
plt.show()


In [ ]:
# then also plot a heatmap with the topic distribution for each document (x axis the topics, y axis the documents)
import seaborn as sns
# plot the document-topic distribution as heatmap
plt.figure(figsize=(10, 15))
sns.heatmap(lda_scratch.doc_topic_dist, cmap='viridis', cbar_kws={'label': 'Topic Probability'})
plt.xlabel('Topics')
#culture mapping to topic
plt.xticks(np.arange(NUM_CULTURES)+0.5, [f"{culture}" for culture in culture_mapping.keys()], rotation=45)
plt.ylabel('Documents')
plt.title('Document-Topic Distribution Heatmap (LDA from Scratch)')
plt.tight_layout()
plt.savefig('./Prior_exploration/document_topic_heatmap.png')
plt.show()

In [ ]:
lda_scratch_2 = LDA_manual(n_topics=2, n_iter=100, random_state=42)
lda_scratch_2.fit(X_resampled_3)


In [ ]:

phi = lda_scratch.topic_word_dist


# empirical distributions for each culture
FBW_triplets = []
triplets_per_vessel = []
for doc in fbw_docs:
    FBW_triplets.extend([triplet for triplet in doc['triplets'] if triplet not in most_common_triplets])
    triplets_per_vessel.append(doc['num_triplets'])
FBW_counts = Counter(FBW_triplets)
FBW_total = sum(FBW_counts.values())
FBW_dist = np.array([FBW_counts.get(triplet, 0) / FBW_total for triplet in features_original])

CW_triplets = []
triplets_per_vessel = []
for doc in cw_docs: 
    CW_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
CW_counts = Counter(CW_triplets)
CW_total = sum(CW_counts.values())
CW_dist = np.array([CW_counts.get(triplet, 0) / CW_total for triplet in features_original])  


plt.figure(figsize=(12, 12))
plt.barh(range(len(features_resampled_3)), FBW_dist, alpha=0.5, label='Empirical FBW Distribution')
plt.barh(range(len(features_resampled_3)), phi[1], alpha=0.5, label='Topic 1')
plt.xlabel('Probability')
plt.yticks(range(len(features_resampled_3)), features_original)
plt.title('Empirical Triplet Distribution for FBW Culture')
plt.legend()
plt.tight_layout()
plt.show()  
plt.figure(figsize=(12, 12))
plt.barh(range(len(features_resampled_3)), CW_dist, alpha=0.5, label='Empirical CW Distribution')
plt.barh(range(len(features_resampled_3)), phi[0], alpha=0.5, label='Topic 0')
plt.xlabel('Probability')
plt.yticks(range(len(features_resampled_3)), features_resampled_3)
plt.title('Empirical Triplet Distribution for CW Culture')
plt.legend()
plt.tight_layout()  
plt.show()  


# Tuning hyperparameters

In [ ]:
empirical_dist_dict = {
    'CW': CW_dist,
    'MI': MI_dist,
    'EFG': EFG_dist,
    'LFG': LFG_dist
}

In [ ]:
def matching_empirical(empirical_dist, topic_dists):
    MI_dist = empirical_dist.get('MI', {})
    CW_dist = empirical_dist.get('CW', {})
    EFG_dist = empirical_dist.get('EFG', {})
    LFG_dist = empirical_dist.get('LFG', {})
    from scipy.spatial.distance import jensenshannon     
    js_MI = [jensenshannon(MI_dist, topic_dists[k])**2 for k in range(NUM_CULTURES)]
    js_CW = [jensenshannon(CW_dist, topic_dists[k])**2 for k in range(NUM_CULTURES)]
    js_EFG = [jensenshannon(EFG_dist, topic_dists[k])**2 for k in range(NUM_CULTURES)]
    js_LFG = [jensenshannon(LFG_dist, topic_dists[k])**2 for k in range(NUM_CULTURES)]

    print("\nJensen-Shannon Divergence between empirical and topic distributions:")
    print(f"MI: {js_MI}")
    print(f"CW: {js_CW}")
    print(f"EFG: {js_EFG}")
    print(f"LFG: {js_LFG}") 

    MI_index = np.argmin(js_MI)
    CW_index = np.argmin(js_CW) 
    EFG_index = np.argmin(js_EFG)
    LFG_index = np.argmin(js_LFG)

    culture_mapping = {
        'MI': MI_index,
        'CW': CW_index,
        'EFG': EFG_index,
        'LFG': LFG_index
    }
    return culture_mapping


In [ ]:
lda_param_grid = [
        (0.1, 0.5), # low alpha, low beta
        (0.1, 1.0), # low alpha, uninformative beta
        (0.1, 2.0), # low alpha, high beta
        (0.3, 0.5), # low-higher alpha, low beta 
        (0.5, 1.0), # low-higher alpha, uninformative beta
        (0.5, 2.0), # low-higher alpha, high beta
        (1.0, 0.5), # high alpha, low beta
        (1.0, 1.0), # uninformative alpha, uninformative beta
        (2.0, 2.0)  # high alpha, high beta
    ]

lda_best_fit = np.zeros((len(lda_param_grid), NUM_CULTURES))

for alpha, beta in lda_param_grid:
    print(f"\nFitting LDA with alpha={alpha}, beta={beta}...")
    #lda_scratch_hyper = LatentDirichletAllocation(n_components=NUM_CULTURES, doc_topic_prior=alpha, topic_word_prior=beta, random_state=42)
    #lda_scratch_hyper.fit(X_resampled_3)
    #raw_phi = lda_scratch_hyper.components_  # shape: (n_topics, n_words) --> counting how many times each word is assigned to each topic
    #phi = raw_phi / raw_phi.sum(axis=1, keepdims=True)
    lda_scratch_hyper = LDA_manual(n_topics=NUM_CULTURES, alpha=alpha, beta=beta, n_iter=100, random_state=42)
    lda_scratch_hyper.fit(X_resampled_5, feature_names=features_resampled_5)
    phi = lda_scratch_hyper.topic_word_dist

    culture_mapping = matching_empirical(empirical_dist_dict, phi)
    MI_index = culture_mapping['MI']
    CW_index = culture_mapping['CW']
    EFG_index = culture_mapping['EFG']
    LFG_index = culture_mapping['LFG']
    print(f"Culture to topic mapping: {culture_mapping}")
    lda_best_fit[lda_param_grid.index((alpha, beta))] = [js_divergence(empirical_dist_dict['MI'], phi[MI_index]), js_divergence(empirical_dist_dict['CW'], phi[CW_index]), js_divergence(empirical_dist_dict['EFG'], phi[EFG_index]), js_divergence(empirical_dist_dict['LFG'], phi[LFG_index])]
        
    plt.figure(figsize=(12, 15))
    plt.subplots(figsize=(12, 15))
    plt.barh(range(len(features_original)), MI_dist, alpha=0.5, label='Empirical MI Distribution')
    plt.barh(range(len(features_original)), phi[MI_index], alpha=0.5, label=f'Topic {MI_index}')
    plt.xlabel('Probability')
    plt.yticks(range(len(features_original)), features_original)
    plt.title(f'Empirical Triplet Distribution for MI Culture, alpha={alpha}, beta={beta}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'./Prior_exploration/MI_topic_word_alpha_{alpha}_beta_{beta}.png')
    plt.show()

    plt.figure(figsize=(12, 15))
    plt.barh(range(len(features_original)), CW_dist, alpha=0.5, label='Empirical CW Distribution')
    plt.barh(range(len(features_original)), phi[CW_index], alpha=0.5, label=f'Topic {CW_index}')
    plt.xlabel('Probability')
    plt.yticks(range(len(features_original)), features_original)
    plt.title(f'Empirical Triplet Distribution for CW Culture, alpha={alpha}, beta={beta}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'./Prior_exploration/CW_topic_word_alpha_{alpha}_beta_{beta}.png')
    plt.show()

    plt.figure(figsize=(12, 15))
    plt.barh(range(len(features_original)), EFG_dist, alpha=0.5, label='Empirical EFG Distribution')
    plt.barh(range(len(features_original)), phi[EFG_index], alpha=0.5, label=f'Topic {EFG_index}')
    plt.xlabel('Probability')
    plt.yticks(range(len(features_original)), features_original)
    plt.title(f'Empirical Triplet Distribution for EFG Culture, alpha={alpha}, beta={beta}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'./Prior_exploration/EFG_topic_word_alpha_{alpha}_beta_{beta}.png')
    plt.show()  

    plt.figure(figsize=(12, 15))
    plt.barh(range(len(features_original)), LFG_dist, alpha=0.5, label='Empirical LFG Distribution')
    plt.barh(range(len(features_original)), phi[LFG_index], alpha=0.5, label=f'Topic {LFG_index}')
    plt.xlabel('Probability')
    plt.yticks(range(len(features_original)), features_original)
    plt.title(f'Empirical Triplet Distribution for LFG Culture, alpha={alpha}, beta={beta}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'./Prior_exploration/LFG_topic_word_alpha_{alpha}_beta_{beta}.png')
    plt.show()


In [ ]:
print("\nLDA Best Fit (Jensen-Shannon Divergence) for different alpha, beta:")
print("Rows: different (alpha, beta) pairs")
print("Columns: MI, CW, EFG, LFG")
print(lda_best_fit)


# Model Evaluation

In [ ]:
new_sample_doc = [[ 
    "(Pressure, Wet smoothing, Simple impression)",
    "(Wet smoothing, Simple impression, Leather-hard)",
    "(Simple impression, Leather-hard, Burnishing)",
    "(Leather-hard, Burnishing, Dry)",
    "(Burnishing, Dry, Open firing)"
],  [  
    "(Coiling, Pressure, Scraping)",
    "(Pressure, Scraping, Wet smoothing)",
    "(Scraping, Wet smoothing, Tilted impression)",
    "(Wet smoothing, Tilted impression, Simple incision)",
    "(Tilted impression, Simple incision, Leather-hard)",
    "(Simple incision, Leather-hard, Dry)",
    "(Leather-hard, Dry, Open firing)"
] , [
    "(Pressure, Wet smoothing, Simple impression)",
    "(Wet smoothing, Tilted impression, Simple incision)",
    "(Tilted impression, Simple incision, Leather-hard)",
    "(Simple incision, Leather-hard, Dry)",
    "(Leather-hard, Dry, Open firing)"
] , [
    "(Coiling, Pressure, Scraping)",
    "(Pressure, Scraping, Wet smoothing)",
    "(Wet smoothing, Simple impression, Leather-hard)",
    "(Simple impression, Leather-hard, Burnishing)",
    "(Leather-hard, Burnishing, Dry)",
    "(Burnishing, Dry, Open firing)"
], 
["(Wet clay, Modelling, Coiling)","(Modelling, Coiling, Pressure)","(Coiling, Pressure, Wet smoothing)","(Pressure, Wet smoothing, Leather-hard)","(Wet smoothing, Leather-hard, Burnishing)","(Leather-hard, Burnishing, Dry)","(Burnishing, Dry, Open firing)"]
]

In [ ]:
triplet_pattern = re.compile(r"\([^()]*?,[^()]*?,[^()]*?\)")
def triplet_tokenizer(text):
    return [normalize_triplet(t) for t in triplet_pattern.findall(text)]

def classify_new_vessel(new_triplets, trained_lda, vocab, culture_mapping=None):
    # Vectorize using the model's vocabulary order
    vectorizer = CountVectorizer(
        tokenizer=triplet_tokenizer,
        lowercase=False,
        token_pattern=None,
        vocabulary=list(vocab)
    )
    tmp_trip = [words for words in new_triplets if words in vocab]
    new_triplets = tmp_trip
    vessel_text = ' '.join(new_triplets)
    print("vessel_text", vessel_text)
    X_new = vectorizer.transform([vessel_text])
    print("X_new", X_new)
    topic_probs = trained_lda.transform(X_new)[0]

    dominant_topic = int(topic_probs.argmax())
    max_prob = float(topic_probs.max())
    if culture_mapping:
        predicted_culture = culture_mapping.get(dominant_topic, f"Topic_{dominant_topic}")
    else:
        predicted_culture = f"Topic_{dominant_topic}"
    confidence = 'High' if max_prob >= 0.6 else ('Medium' if max_prob >= 0.4 else 'Low')
    return {
        'predicted_culture': predicted_culture,
        'topic_probabilities': topic_probs,
        'max_probability': max_prob,
        'confidence': confidence,
        'dominant_topic': dominant_topic
    }


# Test
new_vessel = efg_docs[-20:][0]['triplets']
print(new_vessel)
print(culture_mapping)
result = classify_new_vessel(new_vessel, lda_scratch, vocabulary, culture_mapping=culture_mapping)

In [ ]:
vocab_dict = dict(zip(vocabulary, np.arange(len(vocabulary))))

print(vocab_dict)

In [ ]:
print("\nNew vessel classification:")
print(f"Triplets: {new_vessel}")
print(f"Predicted Culture: {result['predicted_culture']}")
print(f"Topic Probabilities: {result['topic_probabilities']}")
print(f"Dominant Topic: {result['dominant_topic']}")
print(f"confidence: {result['confidence']} (max prob: {result['max_probability']:.3f})")

# Topic Modelling as Networks 

Following the idea of Gerlach, Thiago Peixoto et.al article we construct a Bipartite Network  
* Documents --> Nodes in Layer 1
* Words --> Nodes in Layer 2

Then run a hierarchical Stochastic Block Model to get the communities 


In [ ]:
import os
import pylab as plt
%matplotlib inline  

from sbmtm import sbmtm
import graph_tool.all as gt
import networkx as nx
import graph_tool.all as gt 
# Using resampled_5 
B = nx.Graph()
vocab_text = []
vessel_titles = []
for i, vessel in enumerate(resampled_5): # with the crude "all docs" do not works, the resempling is needed to enhance the real frequency of the triplets
    vessel_id = vessel['vessel_id']
    vessel_titles.append(vessel_id)
    B.add_node(vessel_id, bipartite=0)  # Document layer
    triplet_text = [triplet for triplet in vessel['triplets']]
    vocab_text.append(triplet_text)
    for triplet in vessel['triplets']:
        B.add_node(triplet, bipartite=1)  # Triplet layer
        B.add_edge(vessel_id, triplet)


In [ ]:
print(vessel_titles[10])# or create one if not present
print(vocab_text[0])

In [ ]:
model = sbmtm()
model.make_graph(vocab_text, documents=vessel_titles, counts=True)

gt.seed_rng(32) 
model.fit(overlap=False)
model.plot(nedges=1000)
vertex_labels = []
if len(vertex_labels) == 0:
    vertex_labels = model.g.new_vertex_property("string")
    for v in model.g.vertices():
        vertex_labels[v] = model.g.vp['name'][v]
model.plot(
    nedges=1000,   
    vertex_text=model.g.vp.name,
    vertex_font_size=10,
    vertex_text_position=1,
    output_size=(2200, 1800), 
) 


In [ ]:
model_overlap = sbmtm()
model_overlap.make_graph(vocab_text, documents=vessel_titles, counts=False)
gt.seed_rng(32)
model_overlap.fit(overlap=True)
model_overlap.plot(nedges=1000)
# mantain the same vertex labels as before
if len(vertex_labels) == 0:
    vertex_labels = model_overlap.g.new_vertex_property("string")
    for v in model_overlap.g.vertices():
        vertex_labels[v] = model_overlap.g.vp['name'][v]


In [ ]:
import graph_tool.all as gt
import numpy as np
import matplotlib.pyplot as plt

# assume: model_overlap has been fitted with overlap=True
state = model_overlap.state

# choose level in the hierarchy
l = 0
state_l = state.project_level(l).copy(overlap=True)      # overlapping projection
g = model_overlap.g

# contiguous relabeling of blocks (same trick as in sbmtm.print_topics)
b = gt.contiguous_map(state_l.b)
label_map = {}
for v in g.vertices():
    label_map[state_l.b[v]] = b[v]
state_l = state_l.copy(b=b)

# labeled half-edges (block labels for each endpoint of each edge)
state_l_edges = state_l.get_edge_blocks()                # (b_src, b_tgt) per edge

V = model_overlap.get_V()   # number of word nodes
D = model_overlap.get_D()   # number of document nodes
B = state_l.get_nonempty_B()

# membership counts: n_db[d, r] = # half-edges incident on document d with group r
n_db = np.zeros((D, B), dtype=float)

for e in g.edges():
    d = int(e.source())      # documents are 0..D-1 in sbmtm
    _, b_tgt = state_l_edges[e]   # target is word-group, source is document-group
    if d < D:
        n_db[d, b_tgt] += 1.0

# normalize to get soft memberships p(td | d)
row_sums = n_db.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
p_td_d = n_db / row_sums    # shape: (D, B)

# build a vertex property map with membership vectors for graph-tool pies
vp_pie = g.new_vp("vector<double>")
for d in range(D):
    vp_pie[g.vertex(d)] = p_td_d[d, :].tolist()

# layout on document nodes (simple example using sfdp)
pos = gt.sfdp_layout(g)

# draw only documents, with pie-chart nodes encoding overlaps
gt.graph_draw(
    g,
    pos=pos,
    vertex_fill_color=None,
    vertex_shape="pie",
    vertex_pie_fractions=vp_pie,
    vertex_size=20,
    edge_pen_width=0.2,
    output="overlap_docs_level_%d.png" % l,
)
plt.close()


In [ ]:
g = model.g  # graph-tool graph
vertex_labels = g.new_vertex_property("string")

for v in g.vertices():
    vertex_labels[v] = g.vp.name[v]  # This will be the document or word labels
# l=0 is the lowest level (words/docs), l=1 is the first group level


In [ ]:
# Get the group membership distributions
groups = model.get_groups(l=0)
p_td_d = groups['p_td_d'] # Dimensions: [Number of Groups] x [Number of Documents]
print(p_td_d)

In [ ]:
for e in g.edges():
    u, v = int(e.source()), int(e.target())
    print(f"{g.vp.name[e.source()]} <---> {g.vp.name[e.target()]}")


In [ ]:
model.plot(
    nedges=1000,
    vertex_text=model.g.vp.name,
    vertex_font_size=10,
    vertex_text_position=1,
    output_size=(1200, 800), 
)

In [ ]:
# After fitting
state = model.state
B_level0 = state.get_levels()[0].get_nonempty_B()
print(f"Total groups at level 0: {B_level0}")

# Compare with get_groups extraction
groups = model.get_groups(l=0)
print(f"Word-groups (Bw): {groups['Bw']}")
print(f"Doc-groups (Bd): {groups['Bd']}")
print(f"Sum: {groups['Bw'] + groups['Bd']}")



In [ ]:
import pandas as pd

# Get memberships
p_td_d, p_tw_w = model.group_membership(l=0)
groups = model.get_groups(l=0)

# Documents
doc_columns = [f'Doc_Group_{i}' for i in range(groups['Bd'])]
df_docs = pd.DataFrame(p_td_d.T, 
                       index=model.documents, 
                       columns=doc_columns)
print("Document Memberships:")
print(df_docs)

# Words
word_columns = [f'Word_Group_{i}' for i in range(groups['Bw'])]
df_words = pd.DataFrame(p_tw_w.T, 
                        index=model.words, 
                        columns=word_columns)
print("\nWord Memberships:")
print(df_words)

# Save to CSV
df_docs.to_csv('document_memberships.csv')
df_words.to_csv('word_memberships.csv')


In [ ]:
def get_soft_memberships(model, l=0):
    """
    Extract TRUE soft memberships based on edge label counts
    Returns probabilities that sum to 1 for each node
    """
    V = model.get_V()
    D = model.get_D()
    
    g = model.g
    state = model.state
    state_l = state.project_level(l).copy(overlap=True)
    
    # Get edge blocks (labeled half-edges)
    state_l_edges = state_l.get_edge_blocks()
    
    # Get TOTAL number of groups (this includes both word and doc groups!)
    B = state_l.get_B()  # Use get_B() not get_nonempty_B()
    
    # Count edges by label - use full B dimension
    n_wb = np.zeros((V, B))   # word -> word-group edge counts
    n_db = np.zeros((D, B))   # doc -> doc-group edge counts
    n_dbw = np.zeros((D, B))  # doc -> word-group edge counts
    
    counts = 'count' in g.ep.keys()
    if counts:
        eweight = g.ep["count"]
    else:
        eweight = g.new_ep("int", 1)
    
    ze = gt.ungroup_vector_property(state_l_edges, [0, 1])
    for v1, v2, z1, z2, w in g.get_edges([ze[0], ze[1], eweight]):
        n_db[v1, z1] += w       # document edges labeled z1
        n_dbw[v1, z2] += w      # document edges labeled z2 (word-group)
        n_wb[v2 - D, z2] += w   # word edges labeled z2
    
    # Filter to only non-empty groups
    ind_d = np.where(np.sum(n_db, axis=0) > 0)[0]
    ind_w = np.where(np.sum(n_wb, axis=0) > 0)[0]
    ind_w2 = np.where(np.sum(n_dbw, axis=0) > 0)[0]
    
    Bd = len(ind_d)
    Bw = len(ind_w)
    
    # Extract only the relevant columns
    n_db = n_db[:, ind_d]
    n_wb = n_wb[:, ind_w]
    n_dbw = n_dbw[:, ind_w2]
    
    # Compute SOFT probabilities (normalized edge counts)
    # Handle division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        p_tw_w = (n_wb / np.sum(n_wb, axis=1)[:, np.newaxis]).T
        p_td_d = (n_db / np.sum(n_db, axis=1)[:, np.newaxis]).T
        p_tw_d = (n_dbw / np.sum(n_dbw, axis=1)[:, np.newaxis]).T
        
        # Replace NaN with 0 (for nodes with no edges)
        p_tw_w = np.nan_to_num(p_tw_w)
        p_td_d = np.nan_to_num(p_td_d)
        p_tw_d = np.nan_to_num(p_tw_d)
    
    return {
        'p_td_d': p_td_d,  # shape: (Bd, D)
        'p_tw_w': p_tw_w,  # shape: (Bw, V)
        'p_tw_d': p_tw_d,  # shape: (Bw, D)
        'Bd': Bd,
        'Bw': Bw,
        'n_db': n_db,      # raw edge counts
        'n_wb': n_wb,
        'n_dbw': n_dbw,
        'doc_groups': ind_d,
        'word_groups': ind_w
    }

# Usage
soft_memberships = get_soft_memberships(model_overlap, l=0)

print(f"Found {soft_memberships['Bd']} document-groups and {soft_memberships['Bw']} word-groups")
print(f"Document group indices: {soft_memberships['doc_groups']}")
print(f"Word group indices: {soft_memberships['word_groups']}")

# Now you get REAL probabilities
print("\nSoft Word Memberships:")
word_df = pd.DataFrame(
    soft_memberships['p_tw_w'].T,
    index=model.words,
    columns=[f'WordGroup_{i}' for i in range(soft_memberships['Bw'])]
)
print(word_df.head(20))

print("\nSoft Document Memberships:")
doc_df = pd.DataFrame(
    soft_memberships['p_td_d'].T,
    index=model.documents,
    columns=[f'DocGroup_{i}' for i in range(soft_memberships['Bd'])]
)
print(doc_df.head(20))

# Check for actual overlap
print("\n" + "="*60)
print("OVERLAP ANALYSIS")
print("="*60)

# Words with multiple non-trivial memberships
print("\nWords with significant overlap (>10% in multiple groups):")
overlap_count = 0
for i, word in enumerate(model.words):
    probs = soft_memberships['p_tw_w'][:, i]
    significant = probs[probs > 0.1]
    if len(significant) > 1:
        overlap_count += 1
        prob_dict = {f"Group_{j}": f"{p:.3f}" for j, p in enumerate(probs) if p > 0.1}
        print(f"  {word}: {prob_dict}")
        if overlap_count >= 20:  # Limit output
            break

if overlap_count == 0:
    print("  No words with significant overlap found.")
    print("  This suggests the model found hard partitions.")


# Observations: 

Note: this method cluster both words and documents. 
* However, the words have no overlap, i.e. each word is assigned once and only ones to one culture! (no possibility to shared words)
* The documents are nicely separated between cultures WoW (Looks like no mismatch) some hierarchical structures are present in the documents

In [ ]:
p_td_d,p_tw_w = model.group_membership(l=0)

plt.figure(figsize=(15,15))
plt.imshow(p_td_d,origin='lower',aspect='auto',interpolation='none')
plt.title(r'Document group membership $P(bd | d)$')
plt.xlabel('Document d (index)')
plt.ylabel('Document group, bd')
plt.colorbar()
plt.show()

plt.figure(figsize=(15,15))
plt.imshow(p_tw_w,origin='lower',aspect='auto',interpolation='none')
plt.title(r'Word group membership $P(bw | w)$')
plt.xlabel('Word w (index)')
plt.ylabel('Word group, bw')
plt.colorbar()
plt.show()


In [ ]:
word_topic_memb = model.plot_topic_dist(l=0)
# print the clusters of words 
plt.figure(figsize=(15,15))
plt.imshow(p_tw_w,origin='lower',aspect='auto',interpolation='none')
plt.title(r'Word group membership $P(bw | w)$')
plt.xlabel('Word w (index)')
plt.ylabel('Word group, bw')
plt.colorbar()  
plt.show()

In [ ]:
clusters = model.clusters(l=0,n=1000)
for cluster_id, words in clusters.items():
    print(f"Cluster {cluster_id}: {words}")

In [ ]:
# find how many CW, MI, EFG , LFG in each cluster

# get the naname of the vessels in each cluster clusters[0][0][0] as follow, inside the name there is CW or MI or EFG or LFG find string inside a string 
cluster_culture_counts = defaultdict(Counter)
print(clusters)
for cluster_key, cluster in clusters.items():
    for vessel in cluster:
        vessel_name = vessel[0]  # get the list of vessels in this cluster
        if 'CW' in vessel_name:
            cluster_culture_counts[cluster_key]['CW'] += 1
        elif 'MI' in vessel_name:
            cluster_culture_counts[cluster_key]['MI'] += 1
        elif 'EFG' in vessel_name:
            cluster_culture_counts[cluster_key]['EFG'] += 1
        elif 'LFG' in vessel_name:
            cluster_culture_counts[cluster_key]['LFG'] += 1
        else:
            cluster_culture_counts[cluster_key]['Unknown'] += 1
    # print the counts for each cluster
for cluster_id, counts in cluster_culture_counts.items():
    print(f"Cluster {cluster_id}: {dict(counts)}")  
    # print in a txt file
    with open(f'./NetTM/common_topics.txt', 'w') as f:
        f.write(f"\n Cluster {cluster_id}: {dict(counts)}\n")



In [ ]:
p_td_d,p_tw_w = model.group_membership(l=0)

plt.figure(figsize=(15,10))
plt.subplot(121)
plt.imshow(p_td_d,origin='lower',aspect='auto',interpolation='none')
plt.title(r'Document group membership $P(bd | d)$')
plt.xlabel('Document d (index)')
plt.ylabel('Document group, bd')
plt.colorbar()

plt.subplot(122)
plt.imshow(p_tw_w,origin='lower',aspect='auto',interpolation='none')
plt.title(r'Word group membership $P(bw | w)$')
plt.xlabel('Word w (index)')
plt.ylabel('Word group, bw')
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
model.print_summary()

In [ ]:
model_overlap.clusters(l=0,n=1000)

In [ ]:
model.words

In [ ]:
model_overlap.topics(l=0, n=91)

In [ ]:
# Topic distributions for each document
## select a document (by its index)
i_doc = 15
print(model_overlap.documents[i_doc])
## get a list of tuples (topic-index, probability)
model_overlap.topicdist(i_doc,l=0)

In [ ]:
results = model_overlap.get_groups(l=0)


In [ ]:
p_tw_w = results['p_tw_w']
p_td_d = results['p_td_d']
p_tw_d = results['p_tw_d']
p_w_tw = results['p_w_tw']

In [ ]:
# # group membership of each word-node P(t_w | w)
p_tw_w = results['p_tw_w']
plt.figure(figsize=(10, 8))
plt.imshow(p_tw_w, origin='lower', aspect='auto', interpolation='none')
plt.title(r'Word group membership $P(bw | w)$')
plt.xlabel('Word w (index)')    
plt.ylabel('Word group, bw')
plt.colorbar()
plt.savefig('./NetTM/word_group_membership_model_resampled5.png')
plt.show()

p_td_d = results['p_td_d']
plt.figure(figsize=(10, 8))
plt.imshow(p_td_d, origin='lower', aspect='auto', interpolation='none')     
plt.title(r'Document group membership $P(bd | d)$ , the cultures are CW, MI, EFG, LFG in the documents order (100 each)')
plt.xlabel('Document d (index)')
plt.ylabel('Document group, bd')
plt.colorbar()
plt.savefig('./NetTM/document_group_membership_model_resampled5.png')
plt.show()

p_w_tw = results['p_w_tw']
plt.figure(figsize=(10, 20))
plt.imshow(p_w_tw, origin='lower', aspect='auto', interpolation='none')
# plot on y the triplets in model_overlap 
plt.yticks(np.arange(len(model_overlap.words)), model_overlap.words)
plt.title(r'Word membership to groups $P(w | bw)$')
plt.xlabel('Word group, bw')
plt.ylabel('Word w (index)')
plt.colorbar()
plt.savefig('./NetTM/word_membership_to_groups_model_resampled5.png')
plt.show()

p_tw_d = results['p_tw_d']
plt.figure(figsize=(10, 8))
plt.imshow(p_tw_d.T, origin='lower', aspect='auto', interpolation='none')
plt.title(r'Word group distribution per document $P(bw | d)$')
plt.xlabel('Document d (index)')
plt.ylabel('Word group, bw')
plt.colorbar()
plt.savefig('./NetTM/word_group_distribution_per_document_model_resampled5.png')
plt.show()

In [ ]:
model.print_summary()

In [ ]:
model_overlap.print_summary()

In [ ]:
#plt.barh(range(len(features_resampled_15)), FBW_dist, alpha=0.5, label='Empirical FBW Distribution')
# empirical distributions for each culture
MI_triplets = []
triplets_per_vessel = []
for doc in mi_docs:
    MI_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
print("MI total:", MI_triplets)
MI_counts = Counter(MI_triplets)
print("MI counts:", MI_counts)
MI_total = sum(MI_counts.values())
print("MI total count:", MI_total)
MI_dist = np.array([MI_counts.get(triplet, 0) / MI_total for triplet in model.words])
print("MI dist:", MI_dist)

EFG_triplets = []
triplets_per_vessel = []
for doc in efg_docs:
    EFG_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
EFG_counts = Counter(EFG_triplets)
EFG_total = sum(EFG_counts.values())
EFG_dist = np.array([EFG_counts.get(triplet, 0) / EFG_total for triplet in model.words])

LFG_triplets = []
triplets_per_vessel = []
for doc in lfg_docs:
    LFG_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
LFG_counts = Counter(LFG_triplets)
LFG_total = sum(LFG_counts.values())
LFG_dist = np.array([LFG_counts.get(triplet, 0) / LFG_total for triplet in model.words])

CW_triplets = []
triplets_per_vessel = []
for doc in cw_docs:
    CW_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
CW_counts = Counter(CW_triplets)
CW_total = sum(CW_counts.values())
CW_dist = np.array([CW_counts.get(triplet, 0) / CW_total for triplet in model.words])

# words_plot = [model.topics(l=2, n=91)[0][i][0] for i in range(len(model.topics(l=2, n=91)[0]))]
# prob_word_top = [model.topics(l=2, n=91)[0][i][1] for i in range(len(model.topics(l=2, n=91)[0]))]

# from scipy.spatial.distance import jensenshannon    
# def js_divergence(p, q):
#     return jensenshannon(p, q)**2  # square to get the divergence   
# js_MI = [js_divergence(MI_dist, model.topics(l=2, n=91)[k]) for k in range(np.arange(0,4))]
# js_CW = [js_divergence(CW_dist, model.topics(l=2, n=91)[k]) for k in range(np.arange(0,4))]
# js_EFG = [js_divergence(EFG_dist, model.topics(l=2, n=91)[k]) for k in range(np.arange(0,4))]
# js_LFG = [js_divergence(LFG_dist, model.topics(l=2, n=91)[k]) for k in range(np.arange(0,4))]

# print("\nJensen-Shannon Divergence between empirical and topic distributions:")
# print(f"MI: {js_MI}")
# print(f"CW: {js_CW}")
# print(f"EFG: {js_EFG}")
# print(f"LFG: {js_LFG}") 

# MI_index = np.argmin(js_MI)
# CW_index = np.argmin(js_CW) 
# EFG_index = np.argmin(js_EFG)
# LFG_index = np.argmin(js_LFG)

# culture_mapping = {
#     'MI': MI_index,
#     'CW': CW_index,
#     'EFG': EFG_index,
#     'LFG': LFG_index
# }


In [ ]:
# vertex labels
for vertex in model.g.vertices():
    print(model.g.vp.name[vertex])

# edge labels 
for edge in model.g.edges():
    print(model.g.vp.name[edge.source()], "---", model.g.vp.name[edge.target()])

# Combination of topics

Un'idea che mi era venuta, dato che ho cluster di documenti e cluster di parole separati, ho pensato di vedere quali cluster di parole meglio si addicono ad una cultura, poi sottrarre i cluster "comuni" a tutte e vedere le interazioni a due a due

In [ ]:
culture_labels = ['CW','MI', 'EFG', 'LFG']
empirical_matrix = np.array([CW_dist, MI_dist, EFG_dist, LFG_dist])
# mask empirical matrix if entry > 0 put to 1 else 0 and then normalize each row
empirical_matrix = (empirical_matrix > 0).astype(float)
#empirical_matrix /= empirical_matrix.sum(axis=1, keepdims=True)

result_overlap = model.get_groups(l=0)
model_word_membership = result_overlap['p_tw_w']
final_culture_groups = {name: [] for name in culture_labels}
plt.figure(figsize=(10, 15))
# in a subplot plot the empirical matrix and the model_word_membership side by side
plt.subplot(1, 2, 1)
plt.imshow(empirical_matrix.T, origin='lower', aspect='auto', interpolation='none')
plt.yticks(np.arange(len(model.words)), model.words)
plt.title('Empirical Matrix')
plt.ylabel('Triplet (word) index')      
plt.xlabel('Culture index')
plt.subplot(1, 2, 2)
plt.imshow(model_word_membership.T, origin='lower', aspect='auto', interpolation='none')
plt.title('Model Word Membership')  
plt.ylabel('Triplet (word) index')
plt.xlabel('Word group index')
plt.colorbar()
plt.tight_layout()
plt.savefig('./NetTM/empirical_vs_model_word_membership.png')
plt.show()
# print(np.shape(model_word_membership))
# print(empirical_matrix)

import itertools
for i, culture in enumerate(culture_labels):
    final_culture_groups[culture] = []
    best_overlap_score = -1  
    best_group_indices = None
    for r in range(len(model_word_membership) + 1):
        for group_indices in itertools.combinations(range(len(model_word_membership)), r):
            combined_membership = model_word_membership[list(group_indices), :]
            combined_membership_sum = combined_membership.sum(axis=0).astype(int)
            overlap_vec = np.logical_and(empirical_matrix[i], combined_membership_sum).astype(int)
            xor_vec = np.logical_xor(empirical_matrix[i], combined_membership_sum).astype(int)

            tmp_overlap_score = (overlap_vec - xor_vec).sum()
            if tmp_overlap_score > best_overlap_score:
                best_overlap_score = tmp_overlap_score
                best_group_indices = group_indices
    #print('best_group_indices', best_group_indices, 'for culture', culture)
    if best_group_indices is not None:
        final_culture_groups[culture].extend(best_group_indices)
    final_culture_groups[culture] = list(set(final_culture_groups[culture]))
    print(f"Final group indices for culture {culture}: {final_culture_groups[culture]}")

for culture, group_indices in final_culture_groups.items():
    combined_membership = model_word_membership[group_indices, :].sum(axis=0)
    plt.figure(figsize=(12, 15))
    plt.barh(range(len(model.words)), combined_membership, alpha=0.7, label=f'Inferred {culture} Distribution')
    for i, cult in enumerate(culture_labels):
        if cult == culture:
            plt.barh(range(len(model.words)), empirical_matrix[i], alpha=0.5, label=f'Empirical {culture} Distribution')
    plt.xlabel('Probability')
    plt.yticks(range(len(model.words)), model.words)
    plt.title(f'Inferred Triplet Distribution for {culture} Culture')
    plt.legend()
    plt.tight_layout()
    plt.show()





In [ ]:
culture_labels = ['MI', 'CW', 'EFG', 'LFG']
empirical_matrix = np.array([MI_dist, CW_dist, EFG_dist, LFG_dist])
# mask empirical matrix if entry > 0 put to 1 else 0 and then normalize each row
empirical_matrix = (empirical_matrix > 0).astype(float)
empirical_matrix /= empirical_matrix.sum(axis=1, keepdims=True)

def calculate_jsd(dist1, dist2):
    epsilon = 1e-12
    dist1 = np.array(dist1) + epsilon
    dist2 = np.array(dist2) + epsilon
    dist1 /= dist1.sum()
    dist2 /= dist2.sum()
    return jensenshannon(dist1, dist2)**2
def calculate_overlap_wordsets(set1, set2):
    overlap = np.logical_and(set1, set2)
    return len(overlap)


num_sbm_topics = p_tw_w.shape[0]
print(f"Number of SBM topics: {num_sbm_topics}")
all_sbm_indices = list(range(num_sbm_topics))
final_culture_groups = {name: [] for name in culture_labels}


for culture_name in culture_labels:
    emp_dist = empirical_matrix[culture_labels.index(culture_name)]
    
    current_group_indices = []
    best_jsd_so_far = float('inf') 
    
    while True: 
        best_topic_to_add = -1
        jsd_after_best_addition = best_jsd_so_far
        
        candidate_topics = [i for i in all_sbm_indices if i not in current_group_indices]
        if not candidate_topics:
            break 

        for topic_idx in candidate_topics:
            temp_group = current_group_indices + [topic_idx]
            
            combined_dist = p_tw_w[temp_group].mean(axis=0)
            
            current_jsd = calculate_jsd(emp_dist, combined_dist)
            
            if current_jsd < jsd_after_best_addition:
                jsd_after_best_addition = current_jsd
                best_topic_to_add = topic_idx
        
        if best_topic_to_add != -1 and jsd_after_best_addition < best_jsd_so_far:
            current_group_indices.append(best_topic_to_add)
            best_jsd_so_far = jsd_after_best_addition
            print(f"For culture '{culture_name}', added topic {best_topic_to_add}. New JSD: {best_jsd_so_far:.4f}")
        else:
            break
            
    final_culture_groups[culture_name] = current_group_indices

for name, group in final_culture_groups.items():
    print(f"Final Combined Topic for '{name}' is formed from SBM Topics: {sorted(group)}")

final_combined_topics = np.zeros((len(culture_labels), p_tw_w.shape[1]))
greedy_culture_mapping = {}
for i, name in enumerate(culture_labels):
    group_indices = final_culture_groups[name]
    if group_indices:
        mean_dist = p_tw_w[group_indices].mean(axis=0)
        final_combined_topics[i] = mean_dist / mean_dist.sum()
    else: 
        final_combined_topics[i] = np.ones(p_tw_w.shape[1]) / p_tw_w.shape[1]
    greedy_culture_mapping[name] = i

fig, axes = plt.subplots(2, 2, figsize=(20, 22), constrained_layout=True)

all_words = np.array(model.words)

for ax, culture in zip(axes.flat, culture_labels):
    matched_combined_topic_idx = greedy_culture_mapping[culture]
    
    emp_dist = empirical_matrix[culture_labels.index(culture)]
    combined_topic_dist = final_combined_topics[matched_combined_topic_idx]
    

    
    y_pos = np.arange(len(all_words))
    
    ax.barh(y_pos, emp_dist, height=0.4, label=f'Empirical {culture}', alpha=0.7)
    ax.barh(y_pos, combined_topic_dist, height=0.4, label=f'Combined SBM Topic (from {sorted(final_culture_groups[culture])})',alpha=0.5)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(all_words, fontsize=7)
    ax.invert_yaxis()
    ax.set_title(f"{culture} words and combined clusters of words")
    ax.set_xlabel("Probability")
    ax.grid(axis='x', linestyle='--', alpha=0.6)
    ax.legend()

plt.show()

In [ ]:
model.g.list_properties()
print([edge_prop for edge_prop in model.g.ep['count']])
print([(g.vp.name[edge_prop.source()], g.vp.name[edge_prop.target()])for edge_prop in model.g.edges()])

In [ ]:
list_of_sets = [set(v) for v in final_culture_groups.values()]
common_topics = set.intersection(*list_of_sets)
print(f"SBM Topics common to all 4 cultures: {common_topics}")
unique_topics = {name: set(v) - common_topics for name, v in final_culture_groups.items()}
print(f"SBM Topics unique to each culture:")
for name, topics in unique_topics.items():
    print(f"  {name}: {topics}")

In [ ]:
from itertools import combinations
pairwise_common_topics = {}
for (name1, topics1), (name2, topics2) in combinations(unique_topics.items(), 2):
    common = topics1.intersection(topics2)
    pairwise_common_topics[(name1, name2)] = common 
    print(f"Common SBM Topics between {name1} and {name2}: {common}")   

In [ ]:
# print the words in the common topics for each pair
for (name1, name2), topics in pairwise_common_topics.items():
    words_in_common = set()
    for topic in topics:
        topic_dist = p_tw_w[topic]
        top_indices = np.argsort(topic_dist)[::-1][:5]  # top 5 words
        for word_idx in top_indices:
            words_in_common.add(model.words[word_idx])
    print(f"Words in common SBM Topics between {name1} and {name2}: {words_in_common}")

In [ ]:
# write the above information to a text file
with open('./NetTM/common_topics.txt', 'a') as f:
    f.write(f"SBM Topics common to all 4 cultures: {common_topics}\n\n")
    f.write("SBM Topics unique to each culture:\n")
    for name, topics in unique_topics.items():
        f.write(f"  {name}: {topics}\n")
    f.write("\nPairwise common SBM Topics between cultures:\n")
    for (name1, name2), common in pairwise_common_topics.items():
        f.write(f"  {name1} & {name2}: {common}\n")
    for (name1, name2), topics in pairwise_common_topics.items():
        words_in_common = set()
        for topic in topics:
            topic_dist = p_tw_w[topic]
            top_indices = np.argsort(topic_dist)[::-1][:5]  # top 5 words
            for word_idx in top_indices:
                words_in_common.add(model.words[word_idx])
        f.write(f"Words in common SBM Topics between {name1} and {name2}: {words_in_common}\n")

In [ ]:

def plot_document_membership_heatmap(p_td_d, doc_labels, top_n=40, annotate=True):
    prob = np.array(p_td_d, dtype=float)              # shape: (clusters, documents)
    cols = [f"Cluster {i}" for i in range(prob.shape[0])]
    prob = prob / (prob.sum(axis=0, keepdims=True) + 1e-12)
    df_docs = pd.DataFrame(prob.T, index=doc_labels, columns=cols) * 100
    top_docs = df_docs.max(axis=1).nlargest(top_n).index
    df_top = df_docs.loc[top_docs]
    plt.figure(figsize=(12, 0.35 * len(df_top)))
    sns.heatmap(df_top, cmap='Blues', annot=annotate, fmt='.1f', linewidths=0.3,
                cbar_kws={'label': 'Membership (%)'})
    plt.xlabel('SBM document clusters')
    plt.ylabel('Documents')
    plt.tight_layout()
    plt.show()

def plot_word_membership_heatmap(p_tw_w, word_labels, top_n=91, annotate=True):
    prob = np.array(p_tw_w, dtype=float)             # shape: (clusters, words)
    cols = [f"Cluster {i}" for i in range(prob.shape[0])]
    prob = prob / (prob.sum(axis=0, keepdims=True) + 1e-12)
    df_words = pd.DataFrame(prob.T, index=word_labels, columns=cols) 
    top_words = df_words.max(axis=1).nlargest(top_n).index
    df_top = df_words.loc[top_words]
    plt.figure(figsize=(12, 0.35 * len(df_top)))
    sns.heatmap(df_top, cmap='Reds', annot=annotate, fmt='.1f', linewidths=0.3,
                cbar_kws={'label': 'Membership (%)'})
    plt.xlabel('SBM word clusters')
    plt.ylabel('Words')
    plt.tight_layout()
    plt.show()
plot_word_membership_heatmap(p_tw_w, model.words, top_n=91)


In [ ]:
groups = model.get_groups(l=0)
topic_distr_SBM = groups['p_w_tw']
print(topic_distr_SBM)


In [ ]:
plt.imshow(topic_distr_SBM, origin='lower', aspect='auto', interpolation='none')
plt.show()

In [ ]:
# extract the documents in the first cluster 
groups = model.get_groups(l=0)
print(groups.keys())
print(groups['p_w_tw'].shape)
print(groups['p_td_d'].shape)
print(groups['p_td_d'])

In [ ]:
# find the edges of the bipartite graph 
edges = [(int(e.source()), int(e.target())) for e in model.g.edges()]
print(edges[:10])
# print the names of the vertices in each link 
for e in model.g.edges():
    print(f"{model.g.vp.name[e.source()]} <---> {model.g.vp.name[e.target()]}")


In [ ]:
from collections import Counter

def plot_word_cluster_share(p_td_d, docs, vocab, top_words=50, cmap='magma'):
    cluster_ids = np.argmax(p_td_d, axis=0)                # hard assignment
    n_clusters = p_td_d.shape[0]
    word_index = {w: i for i, w in enumerate(vocab)}
    counts = np.zeros((len(vocab), n_clusters), dtype=float)

    for doc_idx, doc in enumerate(docs):
        cluster = cluster_ids[doc_idx]
        #print("doc_idx", doc_idx, "cluster", cluster, "doc", doc)
        triplet_counts = Counter(doc['triplets'])
        for triplet, freq in triplet_counts.items():
            word_pos = word_index.get(triplet)
            if word_pos is not None:
                counts[word_pos, cluster] += freq

    totals = counts.sum(axis=0, keepdims=True) + 1e-12     # avoid division by zero
    percentages = (counts / totals)

    overall_weight = percentages.sum(axis=1)
    top_idx = np.argsort(overall_weight)[::-1][:top_words]
    heatmap_df = pd.DataFrame(
        percentages[top_idx],
        index=np.array(vocab)[top_idx],
        columns=[f"Cluster {i}" for i in range(n_clusters)]
    )

    plt.figure(figsize=(18, 0.35 * len(heatmap_df)))
    sns.heatmap(heatmap_df, annot=True, fmt=".1f", cmap=cmap,
                linewidths=0.4, cbar_kws={'label': 'Word usage (%)'})
    plt.xlabel("SBM document clusters")
    plt.ylabel("Triplets")
    plt.tight_layout()
    plt.show()

print("Resempled 5 documents:", resampled_15)

plot_word_cluster_share(p_td_d, resampled_15, model.words, top_words=40)

In [ ]:
from collections import Counter

def compute_word_cluster_matrix(p_td_d, docs, vocab):
    cluster_ids = np.argmax(p_td_d, axis=0)
    n_clusters = p_td_d.shape[0]
    word_index = {w: i for i, w in enumerate(vocab)}
    counts = np.zeros((len(vocab), n_clusters), dtype=float)
    cluster_label_counts = [Counter() for _ in range(n_clusters)]

    for doc_idx, doc in enumerate(docs):
        cluster = cluster_ids[doc_idx]
        cluster_label_counts[cluster][doc.get('culture', 'Unknown')] += 1
        triplet_counts = Counter(doc['triplets'])
        for triplet, freq in triplet_counts.items():
            word_pos = word_index.get(triplet)
            if word_pos is not None:
                counts[word_pos, cluster] += freq

    cluster_names = []
    for i, label_count in enumerate(cluster_label_counts):
        if label_count:
            majority_label, majority_count = label_count.most_common(1)[0]
            total_docs = sum(label_count.values())
            cluster_names.append(f"{majority_label} (C{i}, {majority_count}/{total_docs})")
        else:
            cluster_names.append(f"Cluster {i}")
    return counts, cluster_names

def plot_word_cluster_share(counts, vocab, cluster_names, top_words=50, cmap='magma'):
    totals = counts.sum(axis=0, keepdims=True) + 1e-12
    percentages = (counts / totals)
    overall_weight = counts.sum(axis=1)
    top_idx = np.argsort(overall_weight)[::-1][:top_words]
    heatmap_df = pd.DataFrame(
        percentages[top_idx],
        index=np.array(vocab)[top_idx],
        columns=cluster_names
    )
    plt.figure(figsize=(18, 0.35 * len(heatmap_df)))
    sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap=cmap,
                linewidths=0.4, cbar_kws={'label': 'Word usage within cluster (%)'})
    plt.xlabel("SBM document clusters")
    plt.ylabel("Triplets")
    plt.tight_layout()
    plt.show()

def plot_word_cluster_across_share(counts, vocab, cluster_names, top_words=50, cmap='viridis'):
    word_totals = counts.sum(axis=1, keepdims=True) + 1e-12
    percentages = (counts / word_totals)
    overall_weight = counts.sum(axis=1)
    top_idx = np.argsort(overall_weight)[::-1][:top_words]
    heatmap_df = pd.DataFrame(
        percentages[top_idx],
        index=np.array(vocab)[top_idx],
        columns=cluster_names
    )
    plt.figure(figsize=(18, 0.35 * len(heatmap_df)))
    sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap=cmap,
                linewidths=0.4, cbar_kws={'label': 'Word usage across clusters (%)'})
    plt.xlabel("SBM document clusters")
    plt.ylabel("Triplets")
    plt.tight_layout()
    plt.show()

counts_matrix, cluster_names = compute_word_cluster_matrix(p_td_d, resampled_15, model.words)
plot_word_cluster_share(counts_matrix, model.words, cluster_names, top_words=40)
plot_word_cluster_across_share(counts_matrix, model.words, cluster_names, top_words=40)

In [ ]:
results = model.get_groups(l=0)
p_tw_w = results['p_tw_w']
p_td_d = results['p_td_d']
p_tw_d = results['p_tw_d']
p_w_tw = results['p_w_tw']

In [ ]:
def plot_word_cluster_share(counts, vocab, cluster_names, top_words=50, cmap='magma', x_label="SBM document clusters", save = 'model'):
    totals = counts.sum(axis=0, keepdims=True) + 1e-12
    percentages = counts / totals
    indices = np.arange(len(vocab))
    if top_words is not None:
        indices = indices[:top_words]
    heatmap_df = pd.DataFrame(
        percentages[indices],
        index=np.array(vocab)[indices],
        columns=cluster_names
    )
    plt.figure(figsize=(18, 0.35 * len(heatmap_df)))
    sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap=cmap,
                linewidths=0.4, cbar_kws={'label': 'Word usage within clusters'})
    plt.xlabel(x_label)
    plt.ylabel("Triplets")
    plt.tight_layout()
    plt.savefig(f'./plots/Word_usage_within_clusters_{save}.png', format='png', dpi=300)
    plt.show()

def plot_word_cluster_across_share(counts, vocab, cluster_names, top_words=50, cmap='viridis', x_label="SBM document clusters", save = 'model'):
    word_totals = counts.sum(axis=1, keepdims=True) + 1e-12
    percentages = counts / word_totals
    indices = np.arange(len(vocab))
    if top_words is not None:
        indices = indices[:top_words]
    heatmap_df = pd.DataFrame(
        percentages[indices],
        index=np.array(vocab)[indices],
        columns=cluster_names
    )
    plt.figure(figsize=(18, 0.35 * len(heatmap_df)))
    sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap=cmap,
                linewidths=0.4, cbar_kws={'label': 'Word usage across groups'})
    plt.xlabel(x_label)
    plt.ylabel("Triplets")
    plt.tight_layout()
    plt.savefig(f'./plots/Word_usage_across_clusters_{save}.png', format='png', dpi=300)
    plt.show()

def compute_word_culture_matrix(docs, vocab, culture_order=None):
    if culture_order is None:
        culture_order = sorted({doc.get('culture', 'Unknown') for doc in docs})
    word_index = {w: i for i, w in enumerate(vocab)}
    counts = np.zeros((len(vocab), len(culture_order)), dtype=float)
    culture_to_id = {c: idx for idx, c in enumerate(culture_order)}

    for doc in docs:
        culture = doc.get('culture', 'Unknown')
        if culture not in culture_to_id:
            continue
        col = culture_to_id[culture]
        triplet_counts = Counter(doc['triplets'])
        for triplet, freq in triplet_counts.items():
            word_pos = word_index.get(triplet)
            if word_pos is not None:
                counts[word_pos, col] += freq
    return counts, culture_order

counts_matrix, cluster_names = compute_word_cluster_matrix(p_td_d, resampled_5, model.words)
plot_word_cluster_share(counts_matrix, model.words, cluster_names, top_words=40)
plot_word_cluster_across_share(counts_matrix, model.words, cluster_names, top_words=40)

emp_counts_matrix, culture_labels = compute_word_culture_matrix(all_docs, model.words, culture_order=['CW', 'EFG', 'MI', 'LFG'])
plot_word_cluster_share(emp_counts_matrix, model.words, culture_labels, top_words=40, cmap='magma', x_label="Cultures", save='empirical')
plot_word_cluster_across_share(emp_counts_matrix, model.words, culture_labels, top_words=40, cmap='viridis', x_label="Cultures", save='empirical')

In [ ]:
culture_labels = ['CW', 'EFG', 'LFG', 'MI']
# do this for each cluster and culture and store and plot the comparison in a bar chart
counts_matrix, cluster_names = compute_word_cluster_matrix(p_td_d, resampled_5, model.words)
percentages_cluster = counts_matrix[:,0 ] / (counts_matrix[:, 0].sum() + 1e-12)

# empirical distributions for each culture
MI_triplets = []
triplets_per_vessel = []
for doc in mi_docs:
    MI_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
MI_counts = Counter(MI_triplets)
MI_total = sum(MI_counts.values())
MI_dist = np.array([MI_counts.get(triplet, 0) / MI_total for triplet in vocabulary])

plt.barh(range(len(vocabulary)), MI_dist)
plt.barh(range(len(vocabulary)), counts_matrix[:, MI_index])
plt.show()

EFG_triplets = []
triplets_per_vessel = []
for doc in efg_docs:
    EFG_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
EFG_counts = Counter(EFG_triplets)
EFG_total = sum(EFG_counts.values())
EFG_dist = np.array([EFG_counts.get(triplet, 0) / EFG_total for triplet in vocabulary])

LFG_triplets = []
triplets_per_vessel = []
for doc in lfg_docs:
    LFG_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
LFG_counts = Counter(LFG_triplets)
LFG_total = sum(LFG_counts.values())
print("LFG total:", LFG_total)
print("LFG counts for original features:", [LFG_counts.get(triplet, 0) for triplet in vocabulary])
LFG_dist = np.array([LFG_counts.get(triplet, 0) / LFG_total for triplet in vocabulary])  

CW_triplets = []
triplets_per_vessel = []
for doc in cw_docs:
    CW_triplets.extend(doc['triplets'])
    triplets_per_vessel.append(doc['num_triplets'])
CW_counts = Counter(CW_triplets)
CW_total = sum(CW_counts.values())
CW_dist = np.array([CW_counts.get(triplet, 0) / CW_total for triplet in vocabulary])



# calculate using the JS divergence the similarity between the empirical distribution and the topic distribution for each culture
# which is the best matching between empirical and topic distribution
from scipy.spatial.distance import jensenshannon    
def js_divergence(p, q):
    return jensenshannon(p, q)**2  # square to get the divergence   
js_MI = [js_divergence(MI_dist, counts_matrix[:,k]) for k in range(NUM_CULTURES)]
js_CW = [js_divergence(CW_dist, counts_matrix[:, k]) for k in range(NUM_CULTURES)]
js_EFG = [js_divergence(EFG_dist, counts_matrix[:, k]) for k in range(NUM_CULTURES)]
js_LFG = [js_divergence(LFG_dist, counts_matrix[:, k]) for k in range(NUM_CULTURES)]

print("\nJensen-Shannon Divergence between empirical and topic distributions:")
print(f"MI: {js_MI}")
print(f"CW: {js_CW}")
print(f"EFG: {js_EFG}")
print(f"LFG: {js_LFG}") 

MI_index = np.argmin(js_MI)
CW_index = np.argmin(js_CW) 
EFG_index = np.argmin(js_EFG)
LFG_index = np.argmin(js_LFG)

culture_mapping = {
    'MI': MI_index,
    'CW': CW_index,
    'EFG': EFG_index,
    'LFG': LFG_index
}

plt.figure(figsize=(12, 15))
plt.barh(range(len(vocabulary)), MI_dist, alpha=0.5, label='Empirical MI Distribution')
plt.barh(range(len(vocabulary)), counts_matrix[:, MI_index], alpha=0.5, label=f'Topic {MI_index}')
plt.xlabel('Probability')
plt.yticks(range(len(vocabulary)), vocabulary)
plt.title(f'Empirical Triplet Distribution for MI Culture alpha')
plt.legend()
plt.tight_layout()
plt.savefig(f'./comparison_MI_dist.png')
plt.show()

plt.figure(figsize=(12, 15))
plt.barh(range(len(vocabulary)), CW_dist, alpha=0.5, label='Empirical CW Distribution')
plt.barh(range(len(vocabulary)), counts_matrix[:, CW_index], alpha=0.5, label=f'Topic {CW_index}')
plt.xlabel('Probability')
plt.yticks(range(len(vocabulary)), vocabulary)
plt.title(f'Empirical Triplet Distribution for CW Culture ')
plt.legend()
plt.tight_layout()
plt.savefig(f'./comparison_CW_dist.png')
plt.show()

plt.figure(figsize=(12, 15))
plt.barh(range(len(vocabulary)), EFG_dist, alpha=0.5, label='Empirical EFG Distribution')
plt.barh(range(len(vocabulary)), counts_matrix[:, EFG_index], alpha=0.5, label=f'Topic {EFG_index}')
plt.xlabel('Probability')
plt.yticks(range(len(vocabulary)), vocabulary)
plt.title(f'Empirical Triplet Distribution for EFG Culture')
plt.legend()
plt.tight_layout()
plt.savefig(f'./comparison_EFG_dist.png')
plt.show()  

plt.figure(figsize=(12, 15))
plt.barh(range(len(vocabulary)), LFG_dist, alpha=0.5, label='Empirical LFG Distribution')
#plt.barh(range(len(vocabulary)), counts_matrix[:, LFG_index], alpha=0.5, label=f'Topic {LFG_index}')
plt.xlabel('Probability')
plt.yticks(range(len(vocabulary)), vocabulary)
plt.title(f'Empirical Triplet Distribution for LFG Culture')
plt.legend()
plt.tight_layout()
plt.savefig(f'./comparison_LFG_dist.png')
plt.show()


# Testing model

In [ ]:
def classify_vessel_with_sbm(vessel_triplets, model, counts_matrix, cluster_names, vocab, verbose=True):
    """
    Classify a vessel using the SBM-derived word-cluster distributions.
    
    Parameters:
    -----------
    vessel_triplets : list
        List of triplet strings for the vessel
    model : sbmtm model
        Fitted SBM topic model
    counts_matrix : numpy.ndarray
        Word-cluster count matrix (shape: n_words x n_clusters)
    cluster_names : list
        Names/labels for each cluster
    vocab : list
        Vocabulary (word list) matching counts_matrix rows
    verbose : bool
        If True, print detailed classification information
        
    Returns:
    --------
    dict with classification results
    """
    # Clean vocabulary - only keep triplets that exist in vocab
    vocab_set = set(vocab)
    word_index = {w: i for i, w in enumerate(vocab)}
    n_clusters = counts_matrix.shape[1]
    
    # Filter vessel triplets to only those in vocabulary
    valid_triplets = [t for t in vessel_triplets if t in vocab_set]
    invalid_triplets = [t for t in vessel_triplets if t not in vocab_set]
    
    if verbose:
        print(f"\n=== Classification Details ===")
        print(f"Total vessel triplets: {len(vessel_triplets)}")
        print(f"Valid triplets (in vocab): {len(valid_triplets)}")
        print(f"Invalid triplets (not in vocab): {len(invalid_triplets)}")
        if invalid_triplets:
            print(f"  Removed: {invalid_triplets[:5]}{'...' if len(invalid_triplets) > 5 else ''}")
        print(f"Number of clusters: {n_clusters}")
        print(f"Cluster names: {cluster_names}")
    
    if len(valid_triplets) == 0:
        print("WARNING: No valid triplets found in vocabulary!")
        return {
            'predicted_cluster': None,
            'cluster_probabilities': {},
            'max_probability': 0.0,
            'confidence': 'None',
            'error': 'No valid triplets'
        }
    
    # Normalize counts_matrix to get P(w|cluster) for each cluster
    cluster_totals = counts_matrix.sum(axis=0, keepdims=True) + 1e-12
    p_w_cluster = counts_matrix / cluster_totals
    
    # Count triplets in the new vessel
    vessel_counts = Counter(valid_triplets)
    
    if verbose:
        print(f"\nVessel triplet counts (top 10):")
        for triplet, count in vessel_counts.most_common(10):
            print(f"  {triplet}: {count}")
    
    # Compute log-likelihood for each cluster
    log_likelihoods = np.zeros(n_clusters)
    
    for triplet, count in vessel_counts.items():
        word_pos = word_index[triplet]  # Already validated to be in vocab
        
        if verbose:
            print(f"\n'{triplet}' (count={count}):")
        
        for c in range(n_clusters):
            prob = p_w_cluster[word_pos, c]
            if prob > 0:
                log_likelihoods[c] += count * np.log(prob)
                if verbose:
                    print(f"  Cluster {c} ({cluster_names[c]}): P(w|c)={prob:.6f}, log_contrib={count * np.log(prob):.4f}")
            else:
                # Small penalty for zero probability
                log_likelihoods[c] += count * np.log(1e-12)
                if verbose:
                    print(f"  Cluster {c} ({cluster_names[c]}): P(w|c)=0.000000 (penalty applied)")
    
    if verbose:
        print(f"\n=== Summary ===")
        print(f"Valid triplets used: {len(valid_triplets)}/{len(vessel_triplets)}")
        print(f"\nLog-likelihoods:")
        for c in range(n_clusters):
            print(f"  {cluster_names[c]}: {log_likelihoods[c]:.4f}")
    
    # Convert to probabilities (softmax)
    log_likelihoods -= log_likelihoods.max()  # numerical stability
    probs = np.exp(log_likelihoods)
    probs /= probs.sum()
    
    if verbose:
        print(f"\nNormalized probabilities:")
        for c in range(n_clusters):
            print(f"  {cluster_names[c]}: {probs[c]:.4f} ({probs[c]*100:.1f}%)")
    
    # Dominant cluster
    dominant_cluster = int(probs.argmax())
    max_prob = float(probs.max())
    
    return {
        'predicted_cluster': cluster_names[dominant_cluster],
        'cluster_probabilities': dict(zip(cluster_names, probs)),
        'max_probability': max_prob,
        'confidence': 'High' if max_prob >= 0.6 else ('Medium' if max_prob >= 0.4 else 'Low'),
        'dominant_cluster_idx': dominant_cluster,
        'log_likelihoods': log_likelihoods,
        'valid_triplets': len(valid_triplets),
        'total_triplets': len(vessel_triplets),
        'invalid_triplets': invalid_triplets
    }


def refit_model_with_new_vessel(original_titles, original_vocab_text, n_init=1, verbose=True):
    """
    Re-fit the SBM model with the original documents only.
    """
    if verbose:
        print(f"\n=== Re-fitting Model ===")
        print(f"Documents: {len(original_titles)}")

    # Fit new model
    model_new = sbmtm()
    model_new.make_graph(original_vocab_text, documents=original_titles, counts=True)

    gt.seed_rng(32) 
    model_new.fit(overlap=False)
    model_new.plot(nedges=1000)
    
    if verbose:
        print("Model fitted.")
    clusters = model_new.clusters(l=0, n=1000)
    for cluster_id, words in clusters.items():
        print(f"Cluster {cluster_id}: {words}")
    
    return model_new



# Test with verbose output
test_vessel = mi_docs[-1]['triplets']
print("Test vessel triplets:", test_vessel[:3], "...")
result = classify_vessel_with_sbm(test_vessel, model, counts_matrix, cluster_names, model.words, verbose=True)

print("\n" + "="*50)
print("FINAL CLASSIFICATION (Probability-based):")
print(f"Predicted cluster: {result['predicted_cluster']}")
print(f"Confidence: {result['confidence']} (prob: {result['max_probability']:.3f})")

# Now refit the model with the new vessel
print("\n" + "="*50)
print("VALIDATING WITH MODEL RE-FITTING:")
import os
import pylab as plt
%matplotlib inline  

from sbmtm import sbmtm
import graph_tool.all as gt
import networkx as nx
import graph_tool.all as gt 
# Using resampled_5 
B = nx.Graph()
vocab_text = []
vessel_titles = []
for i, vessel in enumerate(resampled_5): # with the crude "all docs" do not works, the resempling is needed to enhance the real frequency of the triplets
    vessel_id = vessel['vessel_id']
    vessel_titles.append(vessel_id)
    B.add_node(vessel_id, bipartite=0)  # Document layer
    triplet_text = [triplet for triplet in vessel['triplets']]
    vocab_text.append(triplet_text)
    for triplet in vessel['triplets']:
        B.add_node(triplet, bipartite=1)  # Triplet layer
        B.add_edge(vessel_id, triplet)

# Add external vessel
new_vessel_id = "NEW_VESSEL_TEST"
new_vessel_triplets = test_vessel
vessel_titles.append(new_vessel_id)
vocab_text.append(new_vessel_triplets)
B.add_node(new_vessel_id, bipartite=0)
for triplet in new_vessel_triplets:
    B.add_node(triplet, bipartite=1)
    B.add_edge(new_vessel_id, triplet)

refit_result = refit_model_with_new_vessel(
    vessel_titles[:-1],  # original docs without new vessel
    vocab_text[:-1],     # original triplets without new vessel
    n_init=1,
    verbose=True
)

In [ ]:
# After training your base model
base_model = model 
base_vocab = {w: i for i, w in enumerate(model.words)}
def classify_vessel_simple(vessel_triplets, model, counts_matrix, cluster_names, verbose=True):
    vocab = model.words
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    n_clusters = counts_matrix.shape[1]
    
    # Filter valid triplets
    valid_triplets = [t for t in vessel_triplets if t in word_to_idx]
    
    if verbose:
        print(f"\n{'='*60}")
        print("CLASSIFICATION")
        print(f"{'='*60}")
        print(f"Vessel has {len(vessel_triplets)} triplets")
        print(f"Valid in vocab: {len(valid_triplets)}")
        print(f"Number of clusters: {n_clusters}")
    
    if len(valid_triplets) == 0:
        return {'error': 'No valid triplets', 'predicted_cluster': None}
    
    # Get P(word | cluster) for each cluster
    cluster_totals = counts_matrix.sum(axis=0, keepdims=True) + 1e-12
    p_word_given_cluster = counts_matrix / cluster_totals  # shape: (n_words, n_clusters)
    
    # Count triplets in vessel
    vessel_counts = Counter(valid_triplets)
    
    # Compute log P(vessel | cluster) for each cluster
    log_likelihoods = np.zeros(n_clusters)
    
    for triplet, count in vessel_counts.items():
        word_idx = word_to_idx[triplet]
        
        for c in range(n_clusters):
            prob = p_word_given_cluster[word_idx, c]
            if prob > 0:
                log_likelihoods[c] += count * np.log(prob)
            else:
                log_likelihoods[c] += count * np.log(1e-12)  # penalty
    
    # Convert to probabilities (softmax)
    log_likelihoods -= log_likelihoods.max()
    probs = np.exp(log_likelihoods)
    probs /= probs.sum()
    
    predicted_cluster_idx = int(probs.argmax())
    
    if verbose:
        print(f"\nCluster probabilities:")
        for c in range(n_clusters):
            marker = " ←" if c == predicted_cluster_idx else ""
            print(f"  {cluster_names[c]}: {probs[c]:.4f} ({probs[c]*100:.1f}%){marker}")
        
        # Show what's IN that cluster
        print(f"\n{'='*60}")
        print(f"PREDICTED CLUSTER: {cluster_names[predicted_cluster_idx]}")
        print(f"{'='*60}")
        
        # Get cluster assignments from model
        p_td_d = model.get_groups(l=0)['p_td_d']
        cluster_ids = np.argmax(p_td_d, axis=0)
        
        docs_in_cluster = [model.documents[i] for i in range(len(model.documents)) 
                          if cluster_ids[i] == predicted_cluster_idx]
        
        # Count cultures
        culture_counts = {}
        for doc_name in docs_in_cluster:
            if "RESAMPLED_" in doc_name:
                culture = doc_name.split("_")[1]
                culture_counts[culture] = culture_counts.get(culture, 0) + 1
        
        print(f"Cluster size: {len(docs_in_cluster)} documents")
        print(f"Culture distribution: {culture_counts}")
        print(f"Sample documents (first 10):")
        for doc in docs_in_cluster[:10]:
            print(f"  - {doc}")
    
    return {
        'predicted_cluster': cluster_names[predicted_cluster_idx],
        'predicted_cluster_idx': predicted_cluster_idx,
        'cluster_probabilities': dict(zip(cluster_names, probs)),
        'max_probability': float(probs.max()),
        'confidence': 'High' if probs.max() >= 0.6 else ('Medium' if probs.max() >= 0.4 else 'Low'),
        'valid_triplets': len(valid_triplets),
        'total_triplets': len(vessel_triplets)
    }


# Test with a known vessel
test_vessel = cw_docs[-2]['triplets']
print(f"Test vessel true culture: EFG")
print(f"Test vessel triplets (first 5): {test_vessel[:5]}")

# Get counts_matrix and cluster_names from your model
counts_matrix, cluster_names = compute_word_cluster_matrix(
    p_td_d, resampled_5, model.words
)
# normalize counts_matrix
counts_matrix = counts_matrix + 1e-6  # avoid zeros
for j in range(counts_matrix.shape[1]):
    counts_matrix[:, j] /= counts_matrix[:, j].sum()

# Classify!
result = classify_vessel_simple(
    test_vessel, 
    base_model, 
    counts_matrix, 
    cluster_names, 
    verbose=True
)

print(f"\n{'='*60}")
print("FINAL RESULT")
print(f"{'='*60}")
print(f"Predicted: {result['predicted_cluster']}")
print(f"Confidence: {result['confidence']} ({result['max_probability']:.1%})")
# do it for all docs 
# confusion matrix
confusion_matrix = {}
true_labels = []
predicted_labels = []
for doc in all_docs:
    vessel_triplets = doc['triplets']
    result = classify_vessel_simple(
        vessel_triplets, 
        base_model, 
        counts_matrix, 
        cluster_names, 
        verbose=False
    )
    print(f"Vessel {doc['vessel_id']} (True: {doc.get('culture', 'Unknown')}): Predicted: {result['predicted_cluster']}, Confidence: {result['confidence']} ({result['max_probability']:.1%})")
    true_culture = doc.get('culture', 'Unknown')
    predicted_culture = result['predicted_cluster'].split(' ')[0]  # get the culture part only
    true_labels.append(true_culture)
    predicted_labels.append(predicted_culture)
    if true_culture not in confusion_matrix:
        confusion_matrix[true_culture] = {}
    if predicted_culture not in confusion_matrix[true_culture]:
        confusion_matrix[true_culture][predicted_culture] = 0
    confusion_matrix[true_culture][predicted_culture] += 1  

print("\nConfusion Matrix:")
cultures = sorted(set(true_labels) | set(predicted_labels))
print(f"{'':<10}", end="")
for c in cultures:
    print(f"{c:<10}", end="")
print()
for true_c in cultures:
    print(f"{true_c:<10}", end="")
    for pred_c in cultures:
        count = confusion_matrix.get(true_c, {}).get(pred_c, 0)
        print(f"{count:<10}", end="")
    print()

# Normalize confusion matrix by row totals (true label counts)
print("\n\nNormalized Confusion Matrix (by true label):")
cultures = sorted(set(true_labels) | set(predicted_labels))

# Calculate row totals
row_totals = {}
for true_c in cultures:
    row_totals[true_c] = sum(confusion_matrix.get(true_c, {}).get(pred_c, 0) for pred_c in cultures)

# Print header
print(f"{'':<10}", end="")
for c in cultures:
    print(f"{c:<10}", end="")
print()

# Print normalized values
for true_c in cultures:
    print(f"{true_c:<10}", end="")
    total = row_totals[true_c]
    for pred_c in cultures:
        count = confusion_matrix.get(true_c, {}).get(pred_c, 0)
        normalized = count / total if total > 0 else 0
        print(f"{normalized:<10.3f}", end="")
    print()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def evaluate_classifier_on_all_docs(
    test_docs_dict, 
    model, 
    counts_matrix, 
    cluster_names, 
    verbose=False
):
    
    # Extract culture labels from cluster names (e.g., "CW (C0, 98/100)" -> "CW")
    cluster_to_culture = {}
    for cluster_name in cluster_names:
        culture = cluster_name.split()[0]  # Take first word (culture code)
        cluster_to_culture[cluster_name] = culture
    
    
    true_labels = []
    predicted_labels = []
    predictions = []
    
    for true_culture, docs in test_docs_dict.items():
        print(f"\n{'='*60}")
        print(f"Testing {len(docs)} vessels from {true_culture}")
        print(f"{'='*60}")
        
        for doc in docs:
            vessel_id = doc['vessel_id']
            vessel_triplets = doc['triplets']
            
            # Classify
            result = classify_vessel_simple(
                vessel_triplets, 
                model, 
                counts_matrix, 
                cluster_names, 
                verbose=False
            )
            
            if 'error' in result:
                print(f"  {vessel_id}: SKIPPED (no valid triplets)")
                continue
            
            predicted_cluster = result['predicted_cluster']
            predicted_culture = cluster_to_culture.get(predicted_cluster, 'Unknown')
            confidence = result['confidence']
            max_prob = result['max_probability']
            
            true_labels.append(true_culture)
            predicted_labels.append(predicted_culture)
            predictions.append({
                'vessel_id': vessel_id,
                'true_culture': true_culture,
                'predicted_culture': predicted_culture,
                'predicted_cluster': predicted_cluster,
                'confidence': confidence,
                'max_probability': max_prob,
                'correct': true_culture == predicted_culture
            })
            
            if verbose:
                status = "CORRECT" if true_culture == predicted_culture else "WRONG"
                print(f"  {status} {vessel_id}: {true_culture} → {predicted_culture} ({confidence}, {max_prob:.2%})")
    
    # Compute confusion matrix
    cultures = sorted(set(true_labels))
    cm = confusion_matrix(true_labels, predicted_labels, labels=cultures)
    
    # Compute metrics
    accuracy = accuracy_score(true_labels, predicted_labels)
    report = classification_report(true_labels, predicted_labels, labels=cultures, 
                                   target_names=cultures, output_dict=True)
    
    return {
        'confusion_matrix': cm,
        'cultures': cultures,
        'accuracy': accuracy,
        'classification_report': report,
        'predictions': predictions,
        'true_labels': true_labels,
        'predicted_labels': predicted_labels
    }


def plot_confusion_matrix(cm, cultures, accuracy, save_path=None):
    """
    Plot a nice confusion matrix with percentages.
    """
    # Normalize by row (true labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=cultures, yticklabels=cultures, 
                ax=axes[0], cbar_kws={'label': 'Count'})
    axes[0].set_title(f'Confusion Matrix (Counts)\nAccuracy: {accuracy:.2%}')
    axes[0].set_ylabel('True Culture')
    axes[0].set_xlabel('Predicted Culture')
    
    # Plot 2: Normalized percentages
    sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=cultures, yticklabels=cultures,
                ax=axes[1], cbar_kws={'label': 'Percentage'})
    axes[1].set_title(f'Confusion Matrix (Normalized)\nAccuracy: {accuracy:.2%}')
    axes[1].set_ylabel('True Culture')
    axes[1].set_xlabel('Predicted Culture')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved confusion matrix to {save_path}")
    
    plt.show()


def print_classification_report(report, cultures):
    """
    Print a nicely formatted classification report.
    """
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT")
    print("="*60)
    
    # Per-class metrics
    print(f"{'Culture':<10} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-" * 60)
    
    for culture in cultures:
        metrics = report[culture]
        print(f"{culture:<10} {metrics['precision']:<12.2%} {metrics['recall']:<12.2%} "
              f"{metrics['f1-score']:<12.2%} {int(metrics['support']):<10}")
    
    # Overall metrics
    print("-" * 60)
    avg = report['macro avg']
    print(f"{'Macro Avg':<10} {avg['precision']:<12.2%} {avg['recall']:<12.2%} "
          f"{avg['f1-score']:<12.2%}")
    
    weighted = report['weighted avg']
    print(f"{'Weighted':<10} {weighted['precision']:<12.2%} {weighted['recall']:<12.2%} "
          f"{weighted['f1-score']:<12.2%}")
    
    print(f"\nOverall Accuracy: {report['accuracy']:.2%}")


# ============================================================================
# RUN EVALUATION
# ============================================================================

# Prepare test data (last 2 vessels from each culture)
test_docs = {
    'CW': cw_docs[:],
    'MI': mi_docs[:],
    'EFG': efg_docs[:],
    'LFG': lfg_docs[:]
}

print("EVALUATING CLASSIFIER ON ALL TEST VESSELS")
print(f"Test set: {sum(len(docs) for docs in test_docs.values())} vessels")

# Run evaluation
results = evaluate_classifier_on_all_docs(
    test_docs, 
    base_model, 
    counts_matrix, 
    cluster_names, 
    verbose=True  # Set to True to see per-vessel results
)

# Print confusion matrix
print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
print(pd.DataFrame(
    results['confusion_matrix'], 
    index=results['cultures'], 
    columns=results['cultures']
))

# Print classification report
print_classification_report(results['classification_report'], results['cultures'])

# Plot confusion matrix
plot_confusion_matrix(
    results['confusion_matrix'], 
    results['cultures'], 
    results['accuracy'],
    save_path='./plots/confusion_matrix_sbm_classifier.png'
)

# Show misclassified vessels
print("\n" + "="*60)
print("MISCLASSIFIED VESSELS")
print("="*60)
misclassified = [p for p in results['predictions'] if not p['correct']]
if misclassified:
    for pred in misclassified:
        print(f" WRONG! {pred['vessel_id']}: {pred['true_culture']} → {pred['predicted_culture']} "
              f"({pred['confidence']}, {pred['max_probability']:.2%})")
else:
    print(" No misclassifications!")

# Save detailed results to CSV
predictions_df = pd.DataFrame(results['predictions'])
predictions_df.to_csv('./classification_results.csv', index=False)
print("\n Detailed results saved to './results/classification_results.csv'")

In [ ]:
import graph_tool.all as gt
import numpy as np

def infer_new_doc_membership(model, doc_words, vocab2id, level=0,
                             n_sweeps_burnin=100, n_sweeps_sample=200):
    """
    model     : fitted sbmtm (already trained on base corpus)
    doc_words : list of word strings for the new document
    vocab2id  : dict mapping word -> word-node index (0..V-1), same as training
    level     : hierarchy level l to work on
    """

    g_base = model.g
    V = model.get_V()
    D_old = model.get_D()

    # 1) add new document vertex (index D_old) to the graph and connect to word nodes
    g = g_base  # in-place on model.g
    v_new = g.add_vertex()          # this becomes document D_old
    doc_index_new = int(v_new)      # should be == D_old

    # add edges for each token
    for w in doc_words:
        if w not in vocab2id:
            continue  # or map to UNK
        w_idx = vocab2id[w]        # word node index in 0..V-1
        v_w = g.vertex(V + w_idx)  # sbmtm uses [0..D-1]=docs, [D..D+V-1]=words
        g.add_edge(v_new, v_w)

    # 2) get projected state at level l and freeze all old vertices
    state = model.state
    state_l = state.project_level(level)

    # extend the block state with the new vertex in some initial group (e.g. 0)
    b = state_l.b.a.copy()
    b = np.append(b, 0)  # initial block label for new doc
    state_l = state_l.copy(b=g.new_vp("int", vals=b))

    # 3) MCMC restricted to the new vertex
    #    we will collect marginals for it only
    B = state_l.get_nonempty_B()
    pv = g.new_vp("vector<double>")  # to store membership counts

    # burn-in
    for _ in range(n_sweeps_burnin):
        state_l.mcmc_sweep(vertices=[v_new])

    # sampling
    for _ in range(n_sweeps_sample):
        state_l.mcmc_sweep(vertices=[v_new])
        pv = state_l.collect_vertex_marginals(pv, vertices=[v_new])

    # 4) normalize the marginal for the new doc to get P(td | d_new)
    counts = np.array(pv[v_new])
    if counts.sum() == 0:
        probs = np.ones(B) / B
    else:
        probs = counts / counts.sum()

    # hard cluster = argmax over groups 0..B-1
    hard_label = int(probs.argmax())

    return probs, hard_label
